# Évaluation finale — Test indépendant 2026
## Généralisation hors échantillon du système verrouillé

Objectif : évaluer une seule fois le système final sur l'année 2026,
après verrouillage complet du modèle, de l'architecture et des
hyperparamètres.

Le modèle évalué est celui défini dans :

`models/block14_final_model_lock.joblib`

Aucun résultat obtenu sur 2026 ne sera utilisé pour modifier le modèle.

In [1]:
# ==========================================
# INITIALISATION
# ==========================================

from pathlib import Path

import hashlib
import joblib
import numpy as np
import pandas as pd


PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

MODEL_DIR = PROJECT_DIR / "models"
DATA_DIR = PROJECT_DIR / "data_processed"


FINAL_LOCK_PATH = (
    MODEL_DIR
    / "block14_final_model_lock.joblib"
)

FINAL_LOCK_HASH_PATH = (
    MODEL_DIR
    / "block14_final_model_lock.sha256"
)


print(
    "Project :",
    PROJECT_DIR
)

print(
    "Final lock existe :",
    FINAL_LOCK_PATH.exists()
)

print(
    "Hash existe :",
    FINAL_LOCK_HASH_PATH.exists()
)

Project : C:\Users\thiba\OneDrive\Bureau\Projet energie
Final lock existe : True
Hash existe : True


## Étape 1 — Vérification du Final Model Lock

### Partie A — Mise en pratique

Avant d'ouvrir les données 2026, on vérifie que le fichier de verrouillage
du modèle n'a pas été modifié depuis la fin du Bloc 14.

Cette vérification utilise l'empreinte SHA-256 enregistrée lors du
verrouillage.

In [2]:
# ==========================================
# SHA-256 FUNCTION
# ==========================================

def sha256_file(path):

    hasher = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:

        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b"",
        ):

            hasher.update(
                chunk
            )

    return hasher.hexdigest()

In [3]:
# ==========================================
# VERIFY FINAL LOCK HASH
# ==========================================

stored_hash = (
    FINAL_LOCK_HASH_PATH
    .read_text(
        encoding="utf-8"
    )
    .strip()
)


current_hash = (
    sha256_file(
        FINAL_LOCK_PATH
    )
)


print(
    "Hash enregistré :",
    stored_hash
)

print(
    "Hash actuel     :",
    current_hash
)

print(
    "Identiques      :",
    stored_hash
    ==
    current_hash
)

Hash enregistré : 40fb9b451409f5ee86bc520350a9a107abc29b17020c9de0ae8c36dbbf7c169f
Hash actuel     : 40fb9b451409f5ee86bc520350a9a107abc29b17020c9de0ae8c36dbbf7c169f
Identiques      : True


In [4]:
# ==========================================
# LOAD FINAL MODEL LOCK
# ==========================================

final_lock = joblib.load(
    FINAL_LOCK_PATH
)


final_configuration = (
    final_lock[
        "configuration"
    ]
)


print(
    "Modèle :",
    final_configuration[
        "model"
    ]
)

print(
    "RMSE de référence 2025 :",
    final_configuration[
        "final_rmse_2025"
    ]
)

print(
    "MLflow :",
    final_configuration[
        "mlflow_model"
    ]
)

print(
    "Version MLflow :",
    final_configuration[
        "mlflow_version"
    ]
)

print(
    "Tests passés :",
    final_configuration[
        "tests_passed"
    ]
)

print(
    "2026 utilisé avant lock :",
    final_lock[
        "uses_2026"
    ]
)

Modèle : Ridge + MLP1 + Patch Transformer + Refined Event Expert
RMSE de référence 2025 : 176.94256118746154
MLflow : energy-demand-forecasting-model
Version MLflow : 4
Tests passés : 24
2026 utilisé avant lock : False


In [5]:
# ==========================================
# LOCK VALIDATION
# ==========================================

assert (
    stored_hash
    ==
    current_hash
)


assert (
    final_lock[
        "model_selection_frozen"
    ]
    is True
)


assert (
    final_lock[
        "hyperparameters_frozen"
    ]
    is True
)


assert (
    final_lock[
        "uses_2026"
    ]
    is False
)


assert np.isclose(
    final_configuration[
        "final_rmse_2025"
    ],
    176.94256118746154,
)


assert (
    final_configuration[
        "tests_passed"
    ]
    ==
    24
)


print(
    "Final Model Lock : vérifié"
)

print(
    "Autorisation d'ouverture de 2026 : OK"
)

Final Model Lock : vérifié
Autorisation d'ouverture de 2026 : OK


### Partie C — Analyse et améliorations guidées par la théorie

Le hash SHA-256 garantit que le fichier utilisé pour définir le modèle
final est identique à celui enregistré avant l'ouverture du jeu de test.

Si $L$ désigne le fichier de verrouillage, on vérifie :

$$
SHA256(L_{\mathrm{actuel}})
=
SHA256(L_{\mathrm{verrouillé}}).
$$

Cette étape permet de conserver une séparation claire entre la phase de
sélection du modèle et la phase d'évaluation finale.

À partir de l'étape suivante, les données 2026 pourront être utilisées,
mais uniquement pour mesurer la généralisation du système déjà figé.

Conclusion :

Le Final Model Lock a été vérifié avant toute ouverture du jeu de test
2026.

Le modèle, ses hyperparamètres et son architecture correspondent bien à
la version verrouillée à la fin du Bloc 14.

L'année 2026 peut maintenant être ouverte pour la première évaluation
hors échantillon définitive.

## Étape 2 — Chargement et audit des données 2026

### Partie A — Mise en pratique

Le verrouillage étant validé, l'année 2026 peut maintenant être ouverte.

Cette étape ne calcule encore aucune performance du modèle. Elle vérifie
uniquement :

- la période réellement disponible ;
- le nombre d'observations ;
- la fréquence temporelle ;
- les doublons ;
- les valeurs manquantes ;
- la présence de la cible ;
- la continuité temporelle.

Comme l'année 2026 est encore en cours, l'évaluation finale actuelle porte
sur la partie de 2026 réellement disponible dans la base.

La date de coupure exacte sera enregistrée afin de rendre l'expérience
reproductible.

In [6]:
# ==========================================
# POSTGRESQL ML DATASET
# ==========================================

import sys


if str(PROJECT_DIR) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_DIR)
    )


from src.database.postgres import (
    load_ml_dataset
)

In [7]:
# ==========================================
# POSTGRESQL PASSWORD
# ==========================================

import os
from getpass import getpass


if not os.getenv(
    "POSTGRES_PASSWORD"
):

    os.environ[
        "POSTGRES_PASSWORD"
    ] = getpass(
        "Mot de passe PostgreSQL : "
    )


print(
    "POSTGRES_PASSWORD disponible :",
    bool(
        os.getenv(
            "POSTGRES_PASSWORD"
        )
    )
)

POSTGRES_PASSWORD disponible : True


In [8]:
from src.database.postgres import (
    get_connection
)


with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute(
            "SELECT 1"
        )

        print(
            "PostgreSQL :",
            cur.fetchone()[0]
        )

PostgreSQL : 1


In [9]:
# ==========================================
# LOAD 2026 TEST DATA
# ==========================================

TEST_START = (
    "2026-01-01"
)

TEST_END = (
    "2027-01-01"
)


df_2026 = load_ml_dataset(
    TEST_START,
    TEST_END,
)


print(
    "Shape :",
    df_2026.shape
)

display(
    df_2026.head()
)

Shape : (0, 23)


,timestamp_utc,consumption_mw,consumption_lag_30m,consumption_lag_1h,consumption_lag_24h,consumption_lag_7d,consumption_mean_3h,consumption_mean_24h,consumption_mean_7d,consumption_std_24h,...,temp_lag_7d,temp_mean_24h,local_hour,day_of_week,month,is_weekend,hour_sin,hour_cos,month_sin,month_cos


In [10]:
# ==========================================
# TEMPORAL COLUMN
# ==========================================

possible_time_columns = [
    "timestamp_utc",
    "timestamp",
    "datetime",
    "date",
]


time_column = next(
    (
        col
        for col
        in possible_time_columns
        if col in df_2026.columns
    ),
    None,
)


assert (
    time_column
    is not None
)


df_2026[
    time_column
] = pd.to_datetime(
    df_2026[
        time_column
    ],
    utc=True,
)


df_2026 = (
    df_2026
    .sort_values(
        time_column
    )
    .reset_index(
        drop=True
    )
)


print(
    "Colonne temporelle :",
    time_column
)

Colonne temporelle : timestamp_utc


In [11]:
# ==========================================
# AVAILABLE TEST PERIOD
# ==========================================

test_first_timestamp = (
    df_2026[
        time_column
    ].min()
)


test_last_timestamp = (
    df_2026[
        time_column
    ].max()
)


test_rows = len(
    df_2026
)


print(
    "Première observation :",
    test_first_timestamp
)

print(
    "Dernière observation :",
    test_last_timestamp
)

print(
    "Nombre d'observations :",
    test_rows
)

Première observation : NaT
Dernière observation : NaT
Nombre d'observations : 0


In [12]:
# ==========================================
# DIAGNOSTIC LOAD_ML_DATASET
# ==========================================

import inspect

print(
    inspect.getsource(
        load_ml_dataset
    )
)

def load_ml_dataset(
    start,
    end
):
    """
    Charge le dataset ML entre start et end.
    """

    query = """
        SELECT *

        FROM energy.ml_dataset

        WHERE
            timestamp_utc >= %s

        AND
            timestamp_utc < %s

        ORDER BY timestamp_utc;
    """

    with get_connection() as conn:

        with conn.cursor() as cur:

            cur.execute(
                query,
                (start, end)
            )

            rows = cur.fetchall()

            columns = [
                desc.name
                for desc
                in cur.description
            ]

    return pd.DataFrame(
        rows,
        columns=columns
    )



In [13]:
# ==========================================
# DATABASE TEMPORAL COVERAGE
# ==========================================

from src.database.postgres import (
    get_connection
)

from psycopg import sql


coverage_rows = []


with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT
                table_name,
                column_name
            FROM information_schema.columns
            WHERE table_schema = 'energy'
              AND data_type IN (
                  'timestamp without time zone',
                  'timestamp with time zone',
                  'date'
              )
            ORDER BY
                table_name,
                ordinal_position;
        """)

        temporal_columns = (
            cur.fetchall()
        )


        for (
            table_name,
            column_name,
        ) in temporal_columns:

            query = sql.SQL("""
                SELECT
                    COUNT(*),
                    MIN({column}),
                    MAX({column})
                FROM {schema}.{table};
            """).format(

                column=sql.Identifier(
                    column_name
                ),

                schema=sql.Identifier(
                    "energy"
                ),

                table=sql.Identifier(
                    table_name
                ),
            )


            try:

                cur.execute(
                    query
                )

                count, date_min, date_max = (
                    cur.fetchone()
                )


                coverage_rows.append({
                    "table":
                        table_name,

                    "time_column":
                        column_name,

                    "rows":
                        count,

                    "min_date":
                        date_min,

                    "max_date":
                        date_max,
                })


            except Exception as exc:

                conn.rollback()

                coverage_rows.append({
                    "table":
                        table_name,

                    "time_column":
                        column_name,

                    "rows":
                        None,

                    "min_date":
                        None,

                    "max_date":
                        None,
                })


database_coverage = pd.DataFrame(
    coverage_rows
)


display(
    database_coverage
    .sort_values(
        [
            "table",
            "time_column",
        ]
    )
)

,table,time_column,rows,min_date,max_date
1,consumption,created_at,227567,2026-09-22 14:52:09.826016+00:00,2026-09-22 14:52:09.826016+00:00
0,consumption,timestamp_utc,227567,2013-01-07 23:00:00+00:00,2025-12-31 22:00:00+00:00
2,consumption_2026_eval,timestamp_utc,12911,2026-01-01 00:00:00+00:00,2026-09-26 23:00:00+00:00
3,consumption_eval_source,timestamp_utc,240478,2013-01-07 23:00:00+00:00,2026-09-26 23:00:00+00:00
4,ml_dataset,timestamp_utc,227231,2013-01-14 23:00:00+00:00,2025-12-31 22:00:00+00:00
5,model_metrics,created_at,1,2026-09-22 15:27:01.063134+00:00,2026-09-22 15:27:01.063134+00:00
7,predictions,created_at,17517,2026-09-22 15:27:52.258019+00:00,2026-09-22 15:27:52.258019+00:00
6,predictions,timestamp_utc,17517,2025-01-01 00:00:00+00:00,2025-12-31 22:00:00+00:00
9,weather,created_at,736416,2026-09-22 14:55:18.805605+00:00,2026-09-22 14:55:18.805605+00:00
8,weather,timestamp_utc,736416,2012-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00


## Étape 2bis — Préparation de l'ingestion 2026

L'audit de PostgreSQL montre que les tables de données s'arrêtent au
31 décembre 2025.

L'absence de lignes 2026 ne provient donc pas du modèle ni de la requête
`load_ml_dataset`, mais de l'absence d'ingestion des nouvelles observations.

Avant toute insertion, on doit reconstruire exactement le même pipeline
que celui utilisé jusqu'en 2025.

En particulier, la table météo contient six séries horaires distinctes.
Ces mêmes six séries devront être prolongées en 2026 afin de ne pas
modifier la définition des variables du modèle verrouillé.

In [14]:
# ==========================================
# SOURCE TABLE SCHEMAS
# ==========================================

schema_query = """
    SELECT
        table_name,
        column_name,
        data_type,
        ordinal_position

    FROM information_schema.columns

    WHERE table_schema = 'energy'

      AND table_name IN (
          'consumption',
          'weather'
      )

    ORDER BY
        table_name,
        ordinal_position;
"""


with get_connection() as conn:

    schema_df = pd.read_sql(
        schema_query,
        conn,
    )


display(
    schema_df
)

C:\Users\thiba\AppData\Local\Temp\ipykernel_24320\2564966687.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  schema_df = pd.read_sql(


,table_name,column_name,data_type,ordinal_position
0,consumption,timestamp_utc,timestamp with time zone,1
1,consumption,consumption_mw,double precision,2
2,consumption,created_at,timestamp with time zone,3
3,weather,timestamp_utc,timestamp with time zone,1
4,weather,city,character varying,2
5,weather,temperature,double precision,3
6,weather,humidity,double precision,4
7,weather,wind_speed,double precision,5
8,weather,precipitation,double precision,6
9,weather,created_at,timestamp with time zone,7


In [15]:
# ==========================================
# LAST SOURCE ROWS
# ==========================================

with get_connection() as conn:

    consumption_tail = pd.read_sql(
        """
        SELECT *
        FROM energy.consumption
        ORDER BY timestamp_utc DESC
        LIMIT 10;
        """,
        conn,
    )

    weather_tail = pd.read_sql(
        """
        SELECT *
        FROM energy.weather
        ORDER BY timestamp_utc DESC
        LIMIT 30;
        """,
        conn,
    )


print("CONSOMMATION")
display(
    consumption_tail
)


print("MÉTÉO")
display(
    weather_tail
)

CONSOMMATION


C:\Users\thiba\AppData\Local\Temp\ipykernel_24320\2782305254.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  consumption_tail = pd.read_sql(
C:\Users\thiba\AppData\Local\Temp\ipykernel_24320\2782305254.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  weather_tail = pd.read_sql(


,timestamp_utc,consumption_mw,created_at
0,2025-12-31 22:00:00+00:00,70514.0,2026-09-22 14:52:09.826016+00:00
1,2025-12-31 21:30:00+00:00,68900.0,2026-09-22 14:52:09.826016+00:00
2,2025-12-31 21:00:00+00:00,67808.0,2026-09-22 14:52:09.826016+00:00
3,2025-12-31 20:30:00+00:00,68681.0,2026-09-22 14:52:09.826016+00:00
4,2025-12-31 20:00:00+00:00,69748.0,2026-09-22 14:52:09.826016+00:00
5,2025-12-31 19:30:00+00:00,71228.0,2026-09-22 14:52:09.826016+00:00
6,2025-12-31 19:00:00+00:00,72799.0,2026-09-22 14:52:09.826016+00:00
7,2025-12-31 18:30:00+00:00,73958.0,2026-09-22 14:52:09.826016+00:00
8,2025-12-31 18:00:00+00:00,74072.0,2026-09-22 14:52:09.826016+00:00
9,2025-12-31 17:30:00+00:00,73549.0,2026-09-22 14:52:09.826016+00:00


MÉTÉO


,timestamp_utc,city,temperature,humidity,wind_speed,precipitation,created_at
0,2025-12-31 23:00:00+00:00,Paris,0.00,None,None,None,2026-09-22 14:55:18.805605+00:00
1,2025-12-31 23:00:00+00:00,Marseille,4.40,None,None,None,2026-09-22 14:55:18.805605+00:00
2,2025-12-31 23:00:00+00:00,Lyon,-3.00,None,None,None,2026-09-22 14:55:18.805605+00:00
3,2025-12-31 23:00:00+00:00,Lille,0.70,None,None,None,2026-09-22 14:55:18.805605+00:00
4,2025-12-31 23:00:00+00:00,France,0.10,None,None,None,2026-09-22 14:55:18.805605+00:00
5,2025-12-31 23:00:00+00:00,Bordeaux,-1.60,None,None,None,2026-09-22 14:55:18.805605+00:00
6,2025-12-31 22:00:00+00:00,Paris,0.30,None,None,None,2026-09-22 14:55:18.805605+00:00
7,2025-12-31 22:00:00+00:00,Marseille,4.90,None,None,None,2026-09-22 14:55:18.805605+00:00
8,2025-12-31 22:00:00+00:00,Lyon,-2.50,None,None,None,2026-09-22 14:55:18.805605+00:00
9,2025-12-31 22:00:00+00:00,Lille,0.90,None,None,None,2026-09-22 14:55:18.805605+00:00


In [16]:
# ==========================================
# WEATHER CATEGORICAL COLUMNS
# ==========================================

with get_connection() as conn:

    weather_sample = pd.read_sql(
        """
        SELECT *
        FROM energy.weather
        ORDER BY timestamp_utc DESC
        LIMIT 1000;
        """,
        conn,
    )


for column in weather_sample.columns:

    if column in [
        "timestamp_utc",
        "created_at",
    ]:
        continue

    n_unique = (
        weather_sample[
            column
        ]
        .nunique(
            dropna=True
        )
    )

    if n_unique <= 20:

        print(
            "\n",
            column,
            ":",
            n_unique,
            "valeurs"
        )

        print(
            weather_sample[
                column
            ]
            .dropna()
            .unique()
        )

C:\Users\thiba\AppData\Local\Temp\ipykernel_24320\4227406532.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  weather_sample = pd.read_sql(



 city : 6 valeurs
['Paris' 'Marseille' 'Lyon' 'Lille' 'France' 'Bordeaux']

 humidity : 0 valeurs
[]

 wind_speed : 0 valeurs
[]

 precipitation : 0 valeurs
[]


In [17]:
# ==========================================
# ML DATASET SCHEMA
# ==========================================

with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT
                column_name,
                data_type,
                ordinal_position

            FROM information_schema.columns

            WHERE table_schema = 'energy'
              AND table_name = 'ml_dataset'

            ORDER BY ordinal_position;
        """)

        rows = cur.fetchall()


ml_schema = pd.DataFrame(
    rows,
    columns=[
        "column_name",
        "data_type",
        "ordinal_position",
    ],
)


display(
    ml_schema
)

,column_name,data_type,ordinal_position
0,timestamp_utc,timestamp with time zone,1
1,consumption_mw,double precision,2
2,consumption_lag_30m,double precision,3
3,consumption_lag_1h,double precision,4
4,consumption_lag_24h,double precision,5
5,consumption_lag_7d,double precision,6
6,consumption_mean_3h,double precision,7
7,consumption_mean_24h,double precision,8
8,consumption_mean_7d,double precision,9
9,consumption_std_24h,double precision,10


In [18]:
# ==========================================
# ML DATASET LAST ROWS
# ==========================================

with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT *
            FROM energy.ml_dataset
            ORDER BY timestamp_utc DESC
            LIMIT 10;
        """)

        rows = cur.fetchall()

        columns = [
            desc.name
            for desc
            in cur.description
        ]


ml_tail = pd.DataFrame(
    rows,
    columns=columns,
)


display(
    ml_tail
)

,timestamp_utc,consumption_mw,consumption_lag_30m,consumption_lag_1h,consumption_lag_24h,consumption_lag_7d,consumption_mean_3h,consumption_mean_24h,consumption_mean_7d,consumption_std_24h,...,temp_lag_7d,temp_mean_24h,local_hour,day_of_week,month,is_weekend,hour_sin,hour_cos,month_sin,month_cos
0,2025-12-31 22:00:00+00:00,70514.0,68900.0,67808.0,70902.0,63473.0,69860.666667,70050.416667,65874.008929,3824.494848,...,2.60,1.020833,23.00000000000000000000,3,12,0,-0.258819,0.965926,-2.449294e-16,1.0
1,2025-12-31 21:30:00+00:00,68900.0,67808.0,68681.0,69855.0,62060.0,70703.666667,70070.312500,65853.651786,3820.865136,...,2.96,1.037500,22.50000000000000000000,3,12,0,-0.382683,0.923880,-2.449294e-16,1.0
2,2025-12-31 21:00:00+00:00,67808.0,68681.0,69748.0,69303.0,61254.0,71747.666667,70101.458333,65834.145833,3808.103442,...,2.96,1.054167,22.00000000000000000000,3,12,0,-0.500000,0.866025,-2.449294e-16,1.0
3,2025-12-31 20:30:00+00:00,68681.0,69748.0,71228.0,70509.0,62080.0,72559.000000,70139.541667,65814.500000,3802.732522,...,3.28,1.072500,21.50000000000000000000,3,12,0,-0.608761,0.793353,-2.449294e-16,1.0
4,2025-12-31 20:00:00+00:00,69748.0,71228.0,72799.0,71628.0,62994.0,72997.500000,70178.708333,65794.398810,3808.291564,...,3.28,1.090833,21.00000000000000000000,3,12,0,-0.707107,0.707107,-2.449294e-16,1.0
5,2025-12-31 19:30:00+00:00,71228.0,72799.0,73958.0,73350.0,64390.0,72780.666667,70222.916667,65774.047619,3832.967979,...,3.34,1.115417,20.50000000000000000000,3,12,0,-0.793353,0.608761,-2.449294e-16,1.0
6,2025-12-31 19:00:00+00:00,72799.0,73958.0,74072.0,74739.0,65793.0,72029.166667,70263.333333,65753.196429,3870.751317,...,3.34,1.140000,20.00000000000000000000,3,12,0,-0.866025,0.500000,-2.449294e-16,1.0
7,2025-12-31 18:30:00+00:00,73958.0,74072.0,73549.0,75673.0,67105.0,70980.500000,70299.062500,65732.800595,3913.262493,...,3.98,1.157500,19.50000000000000000000,3,12,0,-0.923880,0.382683,-2.449294e-16,1.0
8,2025-12-31 18:00:00+00:00,74072.0,73549.0,72379.0,75146.0,67855.0,69972.333333,70321.437500,65714.297619,3938.284570,...,3.98,1.175000,19.00000000000000000000,3,12,0,-0.965926,0.258819,-2.449294e-16,1.0
9,2025-12-31 17:30:00+00:00,73549.0,72379.0,69927.0,73847.0,67951.0,69204.000000,70327.645833,65697.636905,3943.711917,...,4.66,1.178750,18.50000000000000000000,3,12,0,-0.991445,0.130526,-2.449294e-16,1.0


In [19]:
# ==========================================
# ML DATASET RELATION TYPE
# ==========================================

with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT
                c.relname,
                c.relkind

            FROM pg_class c

            JOIN pg_namespace n
                ON n.oid = c.relnamespace

            WHERE n.nspname = 'energy'
              AND c.relname = 'ml_dataset';
        """)

        result = cur.fetchone()


print(
    "Relation :",
    result
)

Relation : ('ml_dataset', 'v')


In [20]:
# ==========================================
# ML DATASET SQL DEFINITION
# ==========================================

with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT pg_get_viewdef(
                'energy.ml_dataset'::regclass,
                true
            );
        """)

        definition = cur.fetchone()[0]


print(
    definition
)

 SELECT timestamp_utc,
    consumption_mw,
    consumption_lag_30m,
    consumption_lag_1h,
    consumption_lag_24h,
    consumption_lag_7d,
    consumption_mean_3h,
    consumption_mean_24h,
    consumption_mean_7d,
    consumption_std_24h,
    temperature,
    temp_lag_1h,
    temp_lag_24h,
    temp_lag_7d,
    temp_mean_24h,
    local_hour,
    day_of_week,
    month,
    is_weekend,
    hour_sin,
    hour_cos,
    month_sin,
    month_cos
   FROM energy.ml_features
  WHERE consumption_lag_7d IS NOT NULL AND timestamp_utc < '2026-01-01 00:00:00+00'::timestamp with time zone;


In [21]:
# ==========================================
# EXACT ML DATASET VIEW DEFINITION
# ==========================================

with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT pg_get_viewdef(
                'energy.ml_dataset'::regclass,
                true
            );
        """)

        ml_view_definition = (
            cur.fetchone()[0]
        )


print(
    "\n========== ML_DATASET VIEW ==========\n"
)

print(
    ml_view_definition
)

print(
    "\n=====================================\n"
)


========== ML_DATASET VIEW ==========

 SELECT timestamp_utc,
    consumption_mw,
    consumption_lag_30m,
    consumption_lag_1h,
    consumption_lag_24h,
    consumption_lag_7d,
    consumption_mean_3h,
    consumption_mean_24h,
    consumption_mean_7d,
    consumption_std_24h,
    temperature,
    temp_lag_1h,
    temp_lag_24h,
    temp_lag_7d,
    temp_mean_24h,
    local_hour,
    day_of_week,
    month,
    is_weekend,
    hour_sin,
    hour_cos,
    month_sin,
    month_cos
   FROM energy.ml_features
  WHERE consumption_lag_7d IS NOT NULL AND timestamp_utc < '2026-01-01 00:00:00+00'::timestamp with time zone;




In [22]:
from IPython.display import (
    Code,
    display,
)


display(
    Code(
        ml_view_definition,
        language="sql",
    )
)

SELECT timestamp_utc,
    consumption_mw,
    consumption_lag_30m,
    consumption_lag_1h,
    consumption_lag_24h,
    consumption_lag_7d,
    consumption_mean_3h,
    consumption_mean_24h,
    consumption_mean_7d,
    consumption_std_24h,
    temperature,
    temp_lag_1h,
    temp_lag_24h,
    temp_lag_7d,
    temp_mean_24h,
    local_hour,
    day_of_week,
    month,
    is_weekend,
    hour_sin,
    hour_cos,
    month_sin,
    month_cos
   FROM energy.ml_features
  WHERE consumption_lag_7d IS NOT NULL AND timestamp_utc < '2026-01-01 00:00:00+00'::timestamp with time zone;

In [23]:
# ==========================================
# FINAL FEATURE CHECK
# ==========================================

display(
    ml_tail[
        [
            "timestamp_utc",
            "consumption_mw",
            "temperature",
            "temp_lag_1h",
            "temp_lag_24h",
            "temp_lag_7d",
            "temp_mean_24h",
            "local_hour",
            "day_of_week",
            "month",
        ]
    ]
)

,timestamp_utc,consumption_mw,temperature,temp_lag_1h,temp_lag_24h,temp_lag_7d,temp_mean_24h,local_hour,day_of_week,month
0,2025-12-31 22:00:00+00:00,70514.0,0.42,0.56,0.94,2.60,1.020833,23.00000000000000000000,3,12
1,2025-12-31 21:30:00+00:00,68900.0,0.56,0.88,1.36,2.96,1.037500,22.50000000000000000000,3,12
2,2025-12-31 21:00:00+00:00,67808.0,0.56,0.88,1.36,2.96,1.054167,22.00000000000000000000,3,12
3,2025-12-31 20:30:00+00:00,68681.0,0.88,1.26,1.76,3.28,1.072500,21.50000000000000000000,3,12
4,2025-12-31 20:00:00+00:00,69748.0,0.88,1.26,1.76,3.28,1.090833,21.00000000000000000000,3,12
5,2025-12-31 19:30:00+00:00,71228.0,1.26,2.30,2.44,3.34,1.115417,20.50000000000000000000,3,12
6,2025-12-31 19:00:00+00:00,72799.0,1.26,2.30,2.44,3.34,1.140000,20.00000000000000000000,3,12
7,2025-12-31 18:30:00+00:00,73958.0,2.30,3.94,3.14,3.98,1.157500,19.50000000000000000000,3,12
8,2025-12-31 18:00:00+00:00,74072.0,2.30,3.94,3.14,3.98,1.175000,19.00000000000000000000,3,12
9,2025-12-31 17:30:00+00:00,73549.0,3.94,4.48,4.12,4.66,1.178750,18.50000000000000000000,3,12


In [24]:
# ==========================================
# FINAL FEATURE CHECK
# ==========================================

display(
    ml_tail[
        [
            "timestamp_utc",
            "consumption_mw",
            "temperature",
            "temp_lag_1h",
            "temp_lag_24h",
            "temp_lag_7d",
            "temp_mean_24h",
            "local_hour",
            "day_of_week",
            "month",
        ]
    ]
)

,timestamp_utc,consumption_mw,temperature,temp_lag_1h,temp_lag_24h,temp_lag_7d,temp_mean_24h,local_hour,day_of_week,month
0,2025-12-31 22:00:00+00:00,70514.0,0.42,0.56,0.94,2.60,1.020833,23.00000000000000000000,3,12
1,2025-12-31 21:30:00+00:00,68900.0,0.56,0.88,1.36,2.96,1.037500,22.50000000000000000000,3,12
2,2025-12-31 21:00:00+00:00,67808.0,0.56,0.88,1.36,2.96,1.054167,22.00000000000000000000,3,12
3,2025-12-31 20:30:00+00:00,68681.0,0.88,1.26,1.76,3.28,1.072500,21.50000000000000000000,3,12
4,2025-12-31 20:00:00+00:00,69748.0,0.88,1.26,1.76,3.28,1.090833,21.00000000000000000000,3,12
5,2025-12-31 19:30:00+00:00,71228.0,1.26,2.30,2.44,3.34,1.115417,20.50000000000000000000,3,12
6,2025-12-31 19:00:00+00:00,72799.0,1.26,2.30,2.44,3.34,1.140000,20.00000000000000000000,3,12
7,2025-12-31 18:30:00+00:00,73958.0,2.30,3.94,3.14,3.98,1.157500,19.50000000000000000000,3,12
8,2025-12-31 18:00:00+00:00,74072.0,2.30,3.94,3.14,3.98,1.175000,19.00000000000000000000,3,12
9,2025-12-31 17:30:00+00:00,73549.0,3.94,4.48,4.12,4.66,1.178750,18.50000000000000000000,3,12


In [25]:
# ==========================================
# FULL ML DATASET VIEW DEFINITION
# ==========================================

print(
    "Longueur SQL :",
    len(
        ml_view_definition
    ),
    "caractères"
)


CHUNK_SIZE = 2000


for start in range(
    0,
    len(
        ml_view_definition
    ),
    CHUNK_SIZE,
):

    end = (
        start
        + CHUNK_SIZE
    )

    print(
        f"\n===== SQL {start}:{end} =====\n"
    )

    print(
        ml_view_definition[
            start:end
        ]
    )

Longueur SQL : 584 caractères

===== SQL 0:2000 =====

 SELECT timestamp_utc,
    consumption_mw,
    consumption_lag_30m,
    consumption_lag_1h,
    consumption_lag_24h,
    consumption_lag_7d,
    consumption_mean_3h,
    consumption_mean_24h,
    consumption_mean_7d,
    consumption_std_24h,
    temperature,
    temp_lag_1h,
    temp_lag_24h,
    temp_lag_7d,
    temp_mean_24h,
    local_hour,
    day_of_week,
    month,
    is_weekend,
    hour_sin,
    hour_cos,
    month_sin,
    month_cos
   FROM energy.ml_features
  WHERE consumption_lag_7d IS NOT NULL AND timestamp_utc < '2026-01-01 00:00:00+00'::timestamp with time zone;


In [26]:
# ==========================================
# SAVE VIEW DEFINITION
# ==========================================

VIEW_SQL_PATH = (
    PROJECT_DIR
    / "sql"
    / "ml_dataset_view_definition.sql"
)


VIEW_SQL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


VIEW_SQL_PATH.write_text(
    ml_view_definition,
    encoding="utf-8",
)


print(
    "SQL sauvegardé :",
    VIEW_SQL_PATH
)

SQL sauvegardé : C:\Users\thiba\OneDrive\Bureau\Projet energie\sql\ml_dataset_view_definition.sql


In [27]:
display(
    ml_tail[
        [
            "timestamp_utc",
            "consumption_mw",
            "temperature",
            "temp_lag_1h",
            "temp_lag_24h",
            "temp_lag_7d",
            "temp_mean_24h",
            "local_hour",
            "day_of_week",
            "month",
        ]
    ]
)

,timestamp_utc,consumption_mw,temperature,temp_lag_1h,temp_lag_24h,temp_lag_7d,temp_mean_24h,local_hour,day_of_week,month
0,2025-12-31 22:00:00+00:00,70514.0,0.42,0.56,0.94,2.60,1.020833,23.00000000000000000000,3,12
1,2025-12-31 21:30:00+00:00,68900.0,0.56,0.88,1.36,2.96,1.037500,22.50000000000000000000,3,12
2,2025-12-31 21:00:00+00:00,67808.0,0.56,0.88,1.36,2.96,1.054167,22.00000000000000000000,3,12
3,2025-12-31 20:30:00+00:00,68681.0,0.88,1.26,1.76,3.28,1.072500,21.50000000000000000000,3,12
4,2025-12-31 20:00:00+00:00,69748.0,0.88,1.26,1.76,3.28,1.090833,21.00000000000000000000,3,12
5,2025-12-31 19:30:00+00:00,71228.0,1.26,2.30,2.44,3.34,1.115417,20.50000000000000000000,3,12
6,2025-12-31 19:00:00+00:00,72799.0,1.26,2.30,2.44,3.34,1.140000,20.00000000000000000000,3,12
7,2025-12-31 18:30:00+00:00,73958.0,2.30,3.94,3.14,3.98,1.157500,19.50000000000000000000,3,12
8,2025-12-31 18:00:00+00:00,74072.0,2.30,3.94,3.14,3.98,1.175000,19.00000000000000000000,3,12
9,2025-12-31 17:30:00+00:00,73549.0,3.94,4.48,4.12,4.66,1.178750,18.50000000000000000000,3,12


In [28]:
# ==========================================
# ML_FEATURES RELATION TYPE
# ==========================================

with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT
                c.relname,
                c.relkind

            FROM pg_class c

            JOIN pg_namespace n
                ON n.oid = c.relnamespace

            WHERE n.nspname = 'energy'
              AND c.relname = 'ml_features';
        """)

        ml_features_relation = (
            cur.fetchone()
        )


print(
    "Relation ml_features :",
    ml_features_relation
)

Relation ml_features : ('ml_features', 'm')


In [29]:
# ==========================================
# ML_FEATURES VIEW DEFINITION
# ==========================================

with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT pg_get_viewdef(
                'energy.ml_features'::regclass,
                true
            );
        """)

        ml_features_definition = (
            cur.fetchone()[0]
        )


print(
    "Longueur SQL :",
    len(
        ml_features_definition
    )
)


CHUNK_SIZE = 2000


for start in range(
    0,
    len(
        ml_features_definition
    ),
    CHUNK_SIZE,
):

    end = (
        start
        + CHUNK_SIZE
    )

    print(
        f"\n===== SQL {start}:{end} =====\n"
    )

    print(
        ml_features_definition[
            start:end
        ]
    )

Longueur SQL : 3964

===== SQL 0:2000 =====

 WITH base AS (
         SELECT c.timestamp_utc,
            c.consumption_mw,
            w.temperature
           FROM energy.consumption c
             LEFT JOIN LATERAL ( SELECT w_1.temperature
                   FROM energy.weather w_1
                  WHERE w_1.city::text = 'France'::text AND w_1.timestamp_utc <= c.timestamp_utc
                  ORDER BY w_1.timestamp_utc DESC
                 LIMIT 1) w ON true
          WHERE c.timestamp_utc < '2026-01-01 00:00:00+00'::timestamp with time zone
        ), temporal AS (
         SELECT base.timestamp_utc,
            base.consumption_mw,
            base.temperature,
            lag(base.consumption_mw, 1) OVER w AS consumption_lag_30m,
            lag(base.consumption_mw, 2) OVER w AS consumption_lag_1h,
            lag(base.consumption_mw, 48) OVER w AS consumption_lag_24h,
            lag(base.consumption_mw, 336) OVER w AS consumption_lag_7d,
            avg(base.consumption_mw) 

In [30]:
# ==========================================
# SAVE ML_FEATURES SQL
# ==========================================

ML_FEATURES_SQL_PATH = (
    PROJECT_DIR
    / "sql"
    / "ml_features_view_definition.sql"
)


ML_FEATURES_SQL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


ML_FEATURES_SQL_PATH.write_text(
    ml_features_definition,
    encoding="utf-8",
)


print(
    "SQL sauvegardé :",
    ML_FEATURES_SQL_PATH
)

SQL sauvegardé : C:\Users\thiba\OneDrive\Bureau\Projet energie\sql\ml_features_view_definition.sql


### Observation — protection historique de 2026

La vue `energy.ml_dataset` contient explicitement une condition excluant
toute observation postérieure ou égale au 1er janvier 2026.

Cette restriction correspond à la protection mise en place pendant la
phase de développement afin de conserver 2026 comme jeu de test final.

Cette vue ne sera pas modifiée.

Une vue distincte dédiée à l'évaluation finale sera créée à partir de
`energy.ml_features`, après vérification complète de son mécanisme de
construction.

### Partie A — Mise en pratique

## Étape 3 — Préparation des sources 2026

Les transformations historiques ont maintenant été identifiées.

La météo utilisée par le modèle correspond à la moyenne des températures
horaires de cinq villes françaises :

- Paris ;
- Lille ;
- Lyon ;
- Marseille ;
- Bordeaux.

Les séries sont obtenues avec Meteostat à partir de stations proches et
interpolées aux coordonnées utilisées pendant le développement.

Avant toute insertion dans PostgreSQL, les données 2026 sont construites
hors base puis comparées aux dernières observations historiques.

In [31]:
# ==========================================
# METEOSTAT 2026
# ==========================================

from datetime import datetime

import meteostat as ms
import numpy as np
import pandas as pd


CITY_POINTS = {

    "Paris":
        (48.86, 2.35),

    "Lille":
        (50.63, 3.06),

    "Lyon":
        (45.76, 4.84),

    "Marseille":
        (43.30, 5.37),

    "Bordeaux":
        (44.84, -0.58),
}


# On garde quelques heures de 2025
# pour vérifier la continuité avec la base.
WEATHER_START = datetime(
    2025,
    12,
    31,
    0,
    0,
)

WEATHER_END = datetime(
    2026,
    9,
    26,
    23,
    0,
)


city_temperature = {}


for city, (
    latitude,
    longitude,
) in CITY_POINTS.items():

    print(
        "Téléchargement :",
        city
    )

    point = ms.Point(
        latitude,
        longitude,
    )

    stations = (
        ms.stations.nearby(
            point,
            limit=4,
        )
    )

    ts = ms.hourly(
        stations,
        WEATHER_START,
        WEATHER_END,
        timezone="UTC",
    )

    df_city = (
        ms.interpolate(
            ts,
            point,
        )
        .fetch()
    )

    if "temp" not in df_city.columns:

        raise ValueError(
            f"Colonne temp absente pour {city}. "
            f"Colonnes : {df_city.columns.tolist()}"
        )

    temperature = (
        df_city[
            "temp"
        ]
        .copy()
    )

    temperature.index = (
        pd.to_datetime(
            temperature.index,
            utc=True,
        )
    )

    city_temperature[
        city
    ] = temperature

Téléchargement : Paris


KeyboardInterrupt: 

In [ ]:
# ==========================================
# FRANCE TEMPERATURE
# ==========================================

temperature_wide = pd.concat(
    city_temperature,
    axis=1,
)


temperature_wide = (
    temperature_wide
    .sort_index()
)


temperature_wide[
    "France"
] = (
    temperature_wide[
        list(
            CITY_POINTS.keys()
        )
    ]
    .mean(
        axis=1
    )
)


display(
    temperature_wide.tail(
        10
    )
)

,Paris,Lille,Lyon,Marseille,Bordeaux,France
time,,,,,,
2026-09-26 14:00:00+00:00,23.3,20.9,27.8,25.5,27.0,24.9
2026-09-26 15:00:00+00:00,23.4,20.2,27.7,25.2,26.9,24.68
2026-09-26 16:00:00+00:00,23.3,19.9,27.7,24.3,27.0,24.44
2026-09-26 17:00:00+00:00,22.7,18.9,26.6,23.0,26.0,23.44
2026-09-26 18:00:00+00:00,21.4,17.9,22.8,21.9,23.3,21.46
2026-09-26 19:00:00+00:00,20.7,16.9,19.3,21.3,21.0,19.84
2026-09-26 20:00:00+00:00,19.9,16.8,18.3,20.7,20.0,19.14
2026-09-26 21:00:00+00:00,19.2,15.7,17.4,20.1,18.6,18.2
2026-09-26 22:00:00+00:00,18.4,14.9,16.6,19.9,18.0,17.56


In [ ]:
# ==========================================
# WEATHER LONG FORMAT
# ==========================================

weather_long = (
    temperature_wide
    .reset_index(
        names="timestamp_utc"
    )
    .melt(
        id_vars="timestamp_utc",
        var_name="city",
        value_name="temperature",
    )
)


weather_long[
    "humidity"
] = np.nan

weather_long[
    "wind_speed"
] = np.nan

weather_long[
    "precipitation"
] = np.nan


weather_long = (
    weather_long
    .sort_values(
        [
            "timestamp_utc",
            "city",
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    weather_long.tail(
        20
    )
)

,timestamp_utc,city,temperature,humidity,wind_speed,precipitation
38860,2026-09-26 20:00:00+00:00,Marseille,20.7,NaN,NaN,NaN
38861,2026-09-26 20:00:00+00:00,Paris,19.9,NaN,NaN,NaN
38862,2026-09-26 21:00:00+00:00,Bordeaux,18.6,NaN,NaN,NaN
38863,2026-09-26 21:00:00+00:00,France,18.2,NaN,NaN,NaN
38864,2026-09-26 21:00:00+00:00,Lille,15.7,NaN,NaN,NaN
38865,2026-09-26 21:00:00+00:00,Lyon,17.4,NaN,NaN,NaN
38866,2026-09-26 21:00:00+00:00,Marseille,20.1,NaN,NaN,NaN
38867,2026-09-26 21:00:00+00:00,Paris,19.2,NaN,NaN,NaN
38868,2026-09-26 22:00:00+00:00,Bordeaux,18.0,NaN,NaN,NaN
38869,2026-09-26 22:00:00+00:00,France,17.56,NaN,NaN,NaN


In [ ]:
# ==========================================
# WEATHER HISTORICAL CHECK
# ==========================================

with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT
                timestamp_utc,
                city,
                temperature

            FROM energy.weather

            WHERE timestamp_utc >=
                '2025-12-31 18:00:00+00'

              AND timestamp_utc <=
                '2025-12-31 23:00:00+00'

            ORDER BY
                timestamp_utc,
                city;
        """)

        rows = cur.fetchall()


weather_db_reference = pd.DataFrame(
    rows,
    columns=[
        "timestamp_utc",
        "city",
        "temperature_db",
    ],
)


weather_api_reference = (
    weather_long[
        (
            weather_long[
                "timestamp_utc"
            ]
            >= pd.Timestamp(
                "2025-12-31 18:00:00+00:00"
            )
        )
        &
        (
            weather_long[
                "timestamp_utc"
            ]
            <= pd.Timestamp(
                "2025-12-31 23:00:00+00:00"
            )
        )
    ]
    [
        [
            "timestamp_utc",
            "city",
            "temperature",
        ]
    ]
    .rename(
        columns={
            "temperature":
                "temperature_new"
        }
    )
)


weather_comparison = (
    weather_db_reference
    .merge(
        weather_api_reference,
        on=[
            "timestamp_utc",
            "city",
        ],
        how="inner",
    )
)


weather_comparison[
    "abs_diff"
] = np.abs(
    weather_comparison[
        "temperature_db"
    ]
    -
    weather_comparison[
        "temperature_new"
    ]
)


display(
    weather_comparison
)


print(
    "Différence maximale :",
    weather_comparison[
        "abs_diff"
    ].max()
)

,timestamp_utc,city,temperature_db,temperature_new,abs_diff
0,2025-12-31 18:00:00+00:00,Bordeaux,1.00,-1.5,2.5
1,2025-12-31 18:00:00+00:00,France,2.30,0.22,2.08
2,2025-12-31 18:00:00+00:00,Lille,2.10,1.2,0.9
3,2025-12-31 18:00:00+00:00,Lyon,-0.80,-3.0,2.2
4,2025-12-31 18:00:00+00:00,Marseille,7.30,4.5,2.8
5,2025-12-31 18:00:00+00:00,Paris,1.90,-0.1,2.0
6,2025-12-31 19:00:00+00:00,Bordeaux,-0.30,-2.0,1.7
7,2025-12-31 19:00:00+00:00,France,1.26,0.06,1.2
8,2025-12-31 19:00:00+00:00,Lille,1.50,1.9,0.4
9,2025-12-31 19:00:00+00:00,Lyon,-1.60,-3.1,1.5


Différence maximale : 2.8


In [ ]:
# ==========================================
# RTE 2026 DOWNLOAD
# ==========================================

import requests


RTE_API = (
    "https://odre.opendatasoft.com/"
    "api/explore/v2.1/catalog/datasets/"
    "eco2mix-national-tr/records"
)


START_2026 = (
    "2026-01-01T00:00:00Z"
)

END_2026 = (
    "2027-01-01T00:00:00Z"
)


records = []

last_timestamp = None


session = requests.Session()


while True:

    where = (
        f"date_heure >= '{START_2026}' "
        f"and date_heure < '{END_2026}'"
    )

    if last_timestamp is not None:

        where += (
            f" and date_heure > "
            f"'{last_timestamp}'"
        )


    params = {

        "select":
            "date_heure,consommation",

        "where":
            where,

        "order_by":
            "date_heure",

        "limit":
            100,
    }


    response = session.get(
        RTE_API,
        params=params,
        timeout=30,
    )

    response.raise_for_status()


    batch = (
        response
        .json()
        .get(
            "results",
            []
        )
    )


    if not batch:

        break


    records.extend(
        batch
    )


    new_last_timestamp = (
        batch[-1][
            "date_heure"
        ]
    )


    if (
        new_last_timestamp
        ==
        last_timestamp
    ):

        break


    last_timestamp = (
        new_last_timestamp
    )


print(
    "Lignes RTE récupérées :",
    len(
        records
    )
)

Lignes RTE récupérées : 8640


In [ ]:
# ==========================================
# RTE CLEANING
# ==========================================

consumption_2026 = pd.DataFrame(
    records
)


consumption_2026[
    "timestamp_utc"
] = pd.to_datetime(
    consumption_2026[
        "date_heure"
    ],
    utc=True,
)


consumption_2026[
    "consumption_mw"
] = pd.to_numeric(
    consumption_2026[
        "consommation"
    ],
    errors="coerce",
)


consumption_2026 = (
    consumption_2026[
        consumption_2026[
            "timestamp_utc"
        ]
        .dt
        .minute
        .isin(
            [
                0,
                30,
            ]
        )
    ]
)


consumption_2026 = (
    consumption_2026
    [
        [
            "timestamp_utc",
            "consumption_mw",
        ]
    ]
    .dropna()
    .groupby(
        "timestamp_utc",
        as_index=False,
    )
    [
        "consumption_mw"
    ]
    .mean()
    .sort_values(
        "timestamp_utc"
    )
    .reset_index(
        drop=True
    )
)


display(
    consumption_2026.head()
)


display(
    consumption_2026.tail()
)

consumption_realtime_2026 = (
    consumption_2026.copy()
)

,timestamp_utc,consumption_mw
0,2026-06-30 22:00:00+00:00,46807.0
1,2026-06-30 22:30:00+00:00,45014.0
2,2026-06-30 23:00:00+00:00,43059.0
3,2026-06-30 23:30:00+00:00,42755.0
4,2026-07-01 00:00:00+00:00,41666.0


,timestamp_utc,consumption_mw
4256,2026-09-27 14:00:00+00:00,38720.0
4257,2026-09-27 14:30:00+00:00,38399.0
4258,2026-09-27 15:00:00+00:00,39048.0
4259,2026-09-27 15:30:00+00:00,39414.0
4260,2026-09-27 16:00:00+00:00,40337.0


In [ ]:
# ==========================================
# RTE 2026 AUDIT
# ==========================================

print(
    "Observations :",
    len(
        consumption_2026
    )
)

print(
    "Début :",
    consumption_2026[
        "timestamp_utc"
    ].min()
)

print(
    "Fin :",
    consumption_2026[
        "timestamp_utc"
    ].max()
)

print(
    "Doublons :",
    consumption_2026[
        "timestamp_utc"
    ]
    .duplicated()
    .sum()
)


rte_steps = (
    consumption_2026[
        "timestamp_utc"
    ]
    .diff()
    .dropna()
)


print(
    "Part des pas 30 min :",
    (
        rte_steps
        ==
        pd.Timedelta(
            minutes=30
        )
    ).mean()
)

Observations : 4261
Début : 2026-06-30 22:00:00+00:00
Fin : 2026-09-27 16:00:00+00:00
Doublons : 0
Part des pas 30 min : 1.0


In [ ]:
# ==========================================
# COMMON 2026 CUTOFF
# ==========================================

weather_2026 = (
    weather_long[
        weather_long[
            "timestamp_utc"
        ].dt.year
        ==
        2026
    ]
)


consumption_last = (
    consumption_2026[
        "timestamp_utc"
    ].max()
)


weather_last = (
    weather_2026[
        "timestamp_utc"
    ].max()
)


COMMON_CUTOFF = min(
    consumption_last,
    weather_last,
)


print(
    "Fin consommation :",
    consumption_last
)

print(
    "Fin météo :",
    weather_last
)

print(
    "Cutoff commun :",
    COMMON_CUTOFF
)

Fin consommation : 2026-09-27 16:00:00+00:00
Fin météo : 2026-09-26 23:00:00+00:00
Cutoff commun : 2026-09-26 23:00:00+00:00


In [ ]:
# ==========================================
# RTE CONSOLIDATED 2026
# ==========================================

def fetch_odre_dataset(
    dataset_id,
    start,
    end,
):

    api_url = (
        "https://odre.opendatasoft.com/"
        "api/explore/v2.1/catalog/datasets/"
        f"{dataset_id}/records"
    )

    records = []
    last_timestamp = None

    while True:

        where = (
            f"date_heure >= '{start}' "
            f"and date_heure < '{end}'"
        )

        if last_timestamp is not None:

            where += (
                f" and date_heure > "
                f"'{last_timestamp}'"
            )

        params = {
            "select":
                "date_heure,consommation",

            "where":
                where,

            "order_by":
                "date_heure",

            "limit":
                100,
        }

        response = session.get(
            api_url,
            params=params,
            timeout=30,
        )

        response.raise_for_status()

        batch = (
            response
            .json()
            .get(
                "results",
                []
            )
        )

        if not batch:
            break

        records.extend(
            batch
        )

        new_last_timestamp = (
            batch[-1][
                "date_heure"
            ]
        )

        if (
            new_last_timestamp
            ==
            last_timestamp
        ):
            break

        last_timestamp = (
            new_last_timestamp
        )

    return pd.DataFrame(
        records
    )

In [ ]:
# ==========================================
# CONSOLIDATED SOURCE
# ==========================================

rte_consolidated_raw = (
    fetch_odre_dataset(
        "eco2mix-national-cons-def",
        "2026-01-01T00:00:00Z",
        "2027-01-01T00:00:00Z",
    )
)


print(
    "Lignes consolidées :",
    len(
        rte_consolidated_raw
    )
)

Lignes consolidées : 17372


In [ ]:
# ==========================================
# CLEAN CONSOLIDATED DATA
# ==========================================

consumption_consolidated_2026 = (
    rte_consolidated_raw.copy()
)


consumption_consolidated_2026[
    "timestamp_utc"
] = pd.to_datetime(
    consumption_consolidated_2026[
        "date_heure"
    ],
    utc=True,
)


consumption_consolidated_2026[
    "consumption_mw"
] = pd.to_numeric(
    consumption_consolidated_2026[
        "consommation"
    ],
    errors="coerce",
)


consumption_consolidated_2026 = (
    consumption_consolidated_2026[
        consumption_consolidated_2026[
            "timestamp_utc"
        ]
        .dt
        .minute
        .isin(
            [0, 30]
        )
    ]
    [
        [
            "timestamp_utc",
            "consumption_mw",
        ]
    ]
    .dropna()
    .groupby(
        "timestamp_utc",
        as_index=False,
    )[
        "consumption_mw"
    ]
    .mean()
)

In [ ]:
# ==========================================
# COMPLETE RTE 2026
# ==========================================

consumption_consolidated_2026[
    "source"
] = "consolidated"


consumption_realtime_2026[
    "source"
] = "realtime"


consumption_complete_2026 = (
    pd.concat(
        [
            consumption_consolidated_2026,
            consumption_realtime_2026,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "timestamp_utc",
            "source",
        ]
    )
)


# En cas de chevauchement,
# on garde la donnée consolidée.
source_priority = {
    "consolidated": 0,
    "realtime": 1,
}


consumption_complete_2026[
    "priority"
] = (
    consumption_complete_2026[
        "source"
    ]
    .map(
        source_priority
    )
)


consumption_complete_2026 = (
    consumption_complete_2026
    .sort_values(
        [
            "timestamp_utc",
            "priority",
        ]
    )
    .drop_duplicates(
        subset="timestamp_utc",
        keep="first",
    )
    .drop(
        columns="priority"
    )
    .reset_index(
        drop=True
    )
)

In [ ]:
# ==========================================
# COMPLETE RTE AUDIT
# ==========================================

rte_diff = (
    consumption_complete_2026[
        "timestamp_utc"
    ]
    .diff()
    .dropna()
)


print(
    "Observations :",
    len(
        consumption_complete_2026
    )
)

print(
    "Début :",
    consumption_complete_2026[
        "timestamp_utc"
    ].min()
)

print(
    "Fin :",
    consumption_complete_2026[
        "timestamp_utc"
    ].max()
)

print(
    "Pas de 30 min :",
    (
        rte_diff
        ==
        pd.Timedelta(
            minutes=30
        )
    ).mean()
)


rte_gaps = (
    rte_diff[
        rte_diff
        >
        pd.Timedelta(
            minutes=30
        )
    ]
)


print(
    "Trous temporels :",
    len(
        rte_gaps
    )
)

Observations : 12945
Début : 2026-01-01 00:00:00+00:00
Fin : 2026-09-27 16:00:00+00:00
Pas de 30 min : 1.0
Trous temporels : 0


In [ ]:
# ==========================================
# METEOSTAT VERSION
# ==========================================

import importlib.metadata


print(
    "Meteostat version :",
    importlib.metadata.version(
        "meteostat"
    )
)

Meteostat version : 2.1.5


In [ ]:
# ==========================================
# HISTORICAL WEATHER REFERENCE
# ==========================================

REFERENCE_START = pd.Timestamp(
    "2025-12-31 18:00:00+00:00"
)

REFERENCE_END = pd.Timestamp(
    "2025-12-31 23:00:00+00:00"
)


reference_cities = (
    weather_db_reference[
        weather_db_reference[
            "city"
        ]
        !=
        "France"
    ]
    .copy()
)

In [ ]:
# ==========================================
# NEARBY STATION BENCHMARK
# ==========================================

station_results = []


for city, (
    latitude,
    longitude,
) in CITY_POINTS.items():

    print(
        "\nVille :",
        city
    )

    point = ms.Point(
        latitude,
        longitude,
    )

    nearby = (
        ms.stations.nearby(
            point,
            limit=8,
        )
    )

    city_reference = (
        reference_cities[
            reference_cities[
                "city"
            ]
            ==
            city
        ]
        [
            [
                "timestamp_utc",
                "temperature_db",
            ]
        ]
    )

    for station_id in nearby.index:

        try:

            ts = ms.hourly(
                str(
                    station_id
                ),
                REFERENCE_START.to_pydatetime(),
                REFERENCE_END.to_pydatetime(),
                timezone="UTC",
            )

            station_df = (
                ts.fetch()
            )

            if (
                station_df.empty
                or
                "temp"
                not in station_df.columns
            ):
                continue

            station_temp = (
                station_df[
                    ["temp"]
                ]
                .reset_index()
            )

            time_col = (
                station_temp.columns[
                    0
                ]
            )

            station_temp = (
                station_temp.rename(
                    columns={
                        time_col:
                            "timestamp_utc",

                        "temp":
                            "temperature_station",
                    }
                )
            )

            station_temp[
                "timestamp_utc"
            ] = pd.to_datetime(
                station_temp[
                    "timestamp_utc"
                ],
                utc=True,
            )

            comparison = (
                city_reference
                .merge(
                    station_temp,
                    on="timestamp_utc",
                    how="inner",
                )
            )

            if comparison.empty:
                continue

            errors = np.abs(
                comparison[
                    "temperature_db"
                ]
                -
                comparison[
                    "temperature_station"
                ]
            )

            station_results.append({
                "city":
                    city,

                "station_id":
                    str(
                        station_id
                    ),

                "n":
                    len(
                        comparison
                    ),

                "mae":
                    float(
                        errors.mean()
                    ),

                "max_diff":
                    float(
                        errors.max()
                    ),
            })

        except Exception:

            continue


station_benchmark = (
    pd.DataFrame(
        station_results
    )
    .sort_values(
        [
            "city",
            "mae",
        ]
    )
)


display(
    station_benchmark
    .groupby(
        "city"
    )
    .head(
        5
    )
)


Ville : Paris

Ville : Lille

Ville : Lyon

Ville : Marseille

Ville : Bordeaux


,city,station_id,n,mae,max_diff
21,Bordeaux,07510,6,1.833333,2.5
7,Lille,07015,6,0.483333,0.9
8,Lille,06414,6,0.633333,0.8
9,Lille,07017,6,0.733333,1.1
11,Lyon,07481,6,1.300000,1.8
13,Lyon,07482,6,1.816667,2.3
10,Lyon,07480,6,1.900000,2.5
12,Lyon,07477,6,3.633333,4.6
14,Lyon,07475,6,4.566667,5.6
16,Marseille,07656,6,0.666667,1.4


### Partie C — Analyse et améliorations guidées par la théorie

L'audit initial de 2026 a révélé deux problèmes de continuité des sources.

La source RTE temps réel ne contient plus l'ensemble des premiers mois
de 2026, car les périodes plus anciennes sont progressivement remplacées
par les données consolidées. L'évaluation doit donc combiner les données
consolidées avec les observations temps réel les plus récentes.

La reconstruction météo actuelle ne reproduit pas suffisamment les valeurs
historiques stockées dans PostgreSQL.

Elle ne peut donc pas encore être utilisée pour l'évaluation finale.

La priorité est la reproductibilité des entrées :

$$
X_{2026}
\sim
\text{même pipeline de construction que } X_{2025}.
$$

Une différence de source ou d'interpolation météo pourrait créer une
rupture artificielle de distribution et fausser la mesure de
généralisation du modèle.

In [ ]:
# ==========================================
# LEGACY METEOSTAT COMPARISON
# ==========================================

legacy_weather = pd.read_csv(
    DATA_DIR
    / "meteostat_legacy_reference.csv"
)


legacy_weather[
    "timestamp_utc"
] = pd.to_datetime(
    legacy_weather[
        "timestamp_utc"
    ],
    utc=True,
)


legacy_comparison = (
    reference_cities
    .merge(
        legacy_weather,
        on=[
            "timestamp_utc",
            "city",
        ],
        how="inner",
    )
)


legacy_comparison[
    "abs_diff"
] = np.abs(
    legacy_comparison[
        "temperature_db"
    ]
    -
    legacy_comparison[
        "temperature"
    ]
)


display(
    legacy_comparison
)


legacy_summary = (
    legacy_comparison
    .groupby(
        "city"
    )
    .agg(
        mae=(
            "abs_diff",
            "mean",
        ),

        max_diff=(
            "abs_diff",
            "max",
        ),

        n=(
            "abs_diff",
            "size",
        ),
    )
    .reset_index()
)


display(
    legacy_summary
)

,timestamp_utc,city,temperature_db,temperature,abs_diff
0,2025-12-31 18:00:00+00:00,Bordeaux,1.0,-1.5,2.5
1,2025-12-31 18:00:00+00:00,Lille,2.1,1.2,0.9
2,2025-12-31 18:00:00+00:00,Lyon,-0.8,-3.0,2.2
3,2025-12-31 18:00:00+00:00,Marseille,7.3,3.7,3.6
4,2025-12-31 18:00:00+00:00,Paris,1.9,-0.1,2.0
5,2025-12-31 19:00:00+00:00,Bordeaux,-0.3,-2.0,1.7
6,2025-12-31 19:00:00+00:00,Lille,1.5,2.0,0.5
7,2025-12-31 19:00:00+00:00,Lyon,-1.6,-3.0,1.4
8,2025-12-31 19:00:00+00:00,Marseille,6.1,3.1,3.0
9,2025-12-31 19:00:00+00:00,Paris,0.6,-0.2,0.8


,city,mae,max_diff,n
0,Bordeaux,1.833333,2.5,6
1,Lille,0.483333,0.9,6
2,Lyon,1.900000,2.5,6
3,Marseille,2.983333,3.6,6
4,Paris,0.700000,2.0,6


### Observation — source de consommation validée

La série RTE 2026 a été reconstruite en combinant les données consolidées
et les observations temps réel récentes.

La série finale contient 12 945 observations entre le 1er janvier 2026
et le 27 septembre 2026, avec un pas strict de 30 minutes et aucun trou
temporel.

La consommation 2026 est donc considérée comme prête pour l'évaluation
finale.

La météo reste temporairement non validée car Meteostat 2.1.5 ne reproduit
pas suffisamment les valeurs historiques stockées dans PostgreSQL.

In [ ]:
# ==========================================
# VERIFY FRANCE TEMPERATURE CONSTRUCTION
# ==========================================

with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            WITH hourly_weather AS (

                SELECT
                    timestamp_utc,

                    MAX(
                        temperature
                    ) FILTER (
                        WHERE city = 'France'
                    ) AS france_db,

                    AVG(
                        temperature
                    ) FILTER (
                        WHERE city <> 'France'
                    ) AS mean_5_cities,

                    COUNT(*) FILTER (
                        WHERE city <> 'France'
                    ) AS n_cities

                FROM energy.weather

                WHERE timestamp_utc >=
                    '2025-01-01 00:00:00+00'

                  AND timestamp_utc <
                    '2026-01-01 00:00:00+00'

                GROUP BY
                    timestamp_utc
            )

            SELECT
                timestamp_utc,
                france_db,
                mean_5_cities,
                n_cities,
                ABS(
                    france_db
                    - mean_5_cities
                ) AS abs_diff

            FROM hourly_weather

            ORDER BY timestamp_utc;
        """)

        rows = cur.fetchall()


france_check = pd.DataFrame(
    rows,
    columns=[
        "timestamp_utc",
        "france_db",
        "mean_5_cities",
        "n_cities",
        "abs_diff",
    ],
)


display(
    france_check.head()
)


print(
    "Observations :",
    len(
        france_check
    )
)

print(
    "Nombre de villes min :",
    france_check[
        "n_cities"
    ].min()
)

print(
    "MAE France vs moyenne :",
    france_check[
        "abs_diff"
    ].mean()
)

print(
    "Écart maximum :",
    france_check[
        "abs_diff"
    ].max()
)

,timestamp_utc,france_db,mean_5_cities,n_cities,abs_diff
0,2025-01-01 00:00:00+00:00,3.28,3.28,5,0.0
1,2025-01-01 01:00:00+00:00,3.12,3.12,5,0.0
2,2025-01-01 02:00:00+00:00,2.88,2.88,5,0.0
3,2025-01-01 03:00:00+00:00,3.26,3.26,5,0.0
4,2025-01-01 04:00:00+00:00,3.30,3.30,5,0.0


Observations : 8760
Nombre de villes min : 5
MAE France vs moyenne : 8.008727971766822e-16
Écart maximum : 7.105427357601002e-15


In [ ]:
# ==========================================
# LOAD HISTORICAL FRANCE WEATHER
# ==========================================

with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT
                timestamp_utc,
                temperature

            FROM energy.weather

            WHERE city = 'France'

              AND timestamp_utc >=
                '2025-01-01 00:00:00+00'

              AND timestamp_utc <
                '2026-01-01 00:00:00+00'

            ORDER BY timestamp_utc;
        """)

        rows = cur.fetchall()


france_db_2025 = pd.DataFrame(
    rows,
    columns=[
        "timestamp_utc",
        "temperature_db",
    ],
)


france_db_2025[
    "timestamp_utc"
] = pd.to_datetime(
    france_db_2025[
        "timestamp_utc"
    ],
    utc=True,
)


print(
    france_db_2025.shape
)

(8760, 2)


In [ ]:
# ==========================================
# METEOSTAT FRANCE FUNCTION
# ==========================================

def build_meteostat_france(
    start,
    end,
):

    temperatures = {}


    for city, (
        latitude,
        longitude,
    ) in CITY_POINTS.items():

        point = ms.Point(
            latitude,
            longitude,
        )

        stations = ms.stations.nearby(
            point,
            limit=4,
        )

        ts = ms.hourly(
            stations,
            start,
            end,
            timezone="UTC",
        )

        data = (
            ms.interpolate(
                ts,
                point,
            )
            .fetch()
        )

        temp = (
            data[
                "temp"
            ]
            .copy()
        )

        temp.index = pd.to_datetime(
            temp.index,
            utc=True,
        )

        temperatures[
            city
        ] = temp


    result = pd.concat(
        temperatures,
        axis=1,
    )


    result[
        "temperature_new"
    ] = (
        result[
            list(
                CITY_POINTS.keys()
            )
        ]
        .mean(
            axis=1
        )
    )


    return (
        result[
            [
                "temperature_new"
            ]
        ]
        .reset_index(
            names="timestamp_utc"
        )
    )

In [ ]:
# ==========================================
# REBUILD 2025 WITH CURRENT METEOSTAT
# ==========================================

france_new_2025 = (
    build_meteostat_france(
        datetime(
            2025,
            1,
            1,
        ),
        datetime(
            2025,
            12,
            31,
            23,
        ),
    )
)


france_comparison_2025 = (
    france_db_2025
    .merge(
        france_new_2025,
        on="timestamp_utc",
        how="inner",
    )
)


france_comparison_2025[
    "error"
] = (
    france_comparison_2025[
        "temperature_db"
    ]
    -
    france_comparison_2025[
        "temperature_new"
    ]
)


france_comparison_2025[
    "abs_error"
] = np.abs(
    france_comparison_2025[
        "error"
    ]
)


print(
    "N comparaison :",
    len(
        france_comparison_2025
    )
)

print(
    "Biais moyen :",
    france_comparison_2025[
        "error"
    ].mean()
)

print(
    "MAE :",
    france_comparison_2025[
        "abs_error"
    ].mean()
)

print(
    "Corrélation :",
    france_comparison_2025[
        [
            "temperature_db",
            "temperature_new",
        ]
    ]
    .corr()
    .iloc[
        0,
        1
    ]
)

N comparaison : 8760
Biais moyen : -0.08998173515981735
MAE : 1.2997397260273973
Corrélation : 0.9727966706318768


In [ ]:
# ==========================================
# WEATHER SOURCE CALIBRATION
# ==========================================

from sklearn.linear_model import (
    LinearRegression
)


weather_adapter = (
    LinearRegression()
)


X_weather = (
    france_comparison_2025[
        [
            "temperature_new"
        ]
    ]
)


y_weather = (
    france_comparison_2025[
        "temperature_db"
    ]
)


weather_adapter.fit(
    X_weather,
    y_weather,
)


print(
    "Intercept :",
    weather_adapter.intercept_
)

print(
    "Coefficient :",
    weather_adapter.coef_[0]
)

Intercept : 0.35227791276408027
Coefficient : 0.9682465924516799


In [ ]:
france_comparison_2025[
    "temperature_calibrated"
] = (
    weather_adapter.predict(
        X_weather
    )
)


calibration_error = (
    france_comparison_2025[
        "temperature_db"
    ]
    -
    france_comparison_2025[
        "temperature_calibrated"
    ]
)


print(
    "RMSE avant calibration :",
    np.sqrt(
        np.mean(
            france_comparison_2025[
                "error"
            ]
            ** 2
        )
    )
)


print(
    "RMSE après calibration :",
    np.sqrt(
        np.mean(
            calibration_error
            ** 2
        )
    )
)


print(
    "MAE après calibration :",
    np.mean(
        np.abs(
            calibration_error
        )
    )
)

RMSE avant calibration : 1.6808674646711306
RMSE après calibration : 1.6627642942768743
MAE après calibration : 1.2852263789007332


### Observation — construction de la température France

L'audit complet de l'année 2025 confirme que la température `France`
stockée dans PostgreSQL correspond exactement à la moyenne arithmétique
des températures de Paris, Lille, Lyon, Marseille et Bordeaux.

Les écarts observés entre la série `France` et la moyenne des cinq villes
sont uniquement de l'ordre de l'erreur numérique en virgule flottante.

La définition historique de la température nationale est donc considérée
comme identifiée.

In [ ]:
# ==========================================
# HISTORICAL FRANCE WEATHER — 2024
# ==========================================

with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT
                timestamp_utc,
                temperature

            FROM energy.weather

            WHERE city = 'France'

              AND timestamp_utc >=
                '2024-01-01 00:00:00+00'

              AND timestamp_utc <
                '2025-01-01 00:00:00+00'

            ORDER BY timestamp_utc;
        """)

        rows = cur.fetchall()


france_db_2024 = pd.DataFrame(
    rows,
    columns=[
        "timestamp_utc",
        "temperature_db",
    ],
)


france_db_2024[
    "timestamp_utc"
] = pd.to_datetime(
    france_db_2024[
        "timestamp_utc"
    ],
    utc=True,
)


print(
    "Ancienne météo 2024 :",
    france_db_2024.shape
)

Ancienne météo 2024 : (8784, 2)


In [ ]:
# ==========================================
# CURRENT METEOSTAT — 2024
# ==========================================

france_new_2024 = (
    build_meteostat_france(
        datetime(
            2024,
            1,
            1,
        ),
        datetime(
            2024,
            12,
            31,
            23,
        ),
    )
)


weather_2024 = (
    france_db_2024
    .merge(
        france_new_2024,
        on="timestamp_utc",
        how="inner",
    )
)


print(
    "Comparaison 2024 :",
    weather_2024.shape
)

Comparaison 2024 : (8784, 3)


In [ ]:
# ==========================================
# WEATHER ADAPTER FEATURES
# ==========================================

def add_weather_adapter_features(
    df
):

    result = df.copy()

    timestamp = (
        result[
            "timestamp_utc"
        ]
    )

    hour = (
        timestamp
        .dt
        .hour
    )

    month = (
        timestamp
        .dt
        .month
    )


    result[
        "hour_sin"
    ] = np.sin(
        2
        * np.pi
        * hour
        / 24
    )


    result[
        "hour_cos"
    ] = np.cos(
        2
        * np.pi
        * hour
        / 24
    )


    result[
        "month_sin"
    ] = np.sin(
        2
        * np.pi
        * month
        / 12
    )


    result[
        "month_cos"
    ] = np.cos(
        2
        * np.pi
        * month
        / 12
    )


    return result

In [ ]:
weather_2024 = (
    add_weather_adapter_features(
        weather_2024
    )
)


weather_2025 = (
    add_weather_adapter_features(
        france_comparison_2025
    )
)

In [ ]:
# ==========================================
# RIDGE WEATHER SOURCE ADAPTER
# ==========================================

from sklearn.linear_model import (
    Ridge
)

from sklearn.pipeline import (
    Pipeline
)

from sklearn.preprocessing import (
    StandardScaler
)


WEATHER_FEATURES = [
    "temperature_new",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
]


weather_source_adapter = Pipeline([
    (
        "scaler",
        StandardScaler(),
    ),

    (
        "ridge",
        Ridge(
            alpha=1.0
        ),
    ),
])


weather_source_adapter.fit(
    weather_2024[
        WEATHER_FEATURES
    ],
    weather_2024[
        "temperature_db"
    ],
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('ridge', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['temperature_new','hour_sin','hour_cos','month_sin','month_cos']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [ ]:
# ==========================================
# OUT-OF-SAMPLE WEATHER VALIDATION — 2025
# ==========================================

weather_2025[
    "temperature_adapter"
] = (
    weather_source_adapter.predict(
        weather_2025[
            WEATHER_FEATURES
        ]
    )
)


error_raw = (
    weather_2025[
        "temperature_db"
    ]
    -
    weather_2025[
        "temperature_new"
    ]
)


error_adapter = (
    weather_2025[
        "temperature_db"
    ]
    -
    weather_2025[
        "temperature_adapter"
    ]
)


rmse_raw_2025 = np.sqrt(
    np.mean(
        error_raw ** 2
    )
)


rmse_adapter_2025 = np.sqrt(
    np.mean(
        error_adapter ** 2
    )
)


mae_raw_2025 = np.mean(
    np.abs(
        error_raw
    )
)


mae_adapter_2025 = np.mean(
    np.abs(
        error_adapter
    )
)


print(
    "RMSE brut 2025 :",
    rmse_raw_2025
)

print(
    "RMSE adaptateur 2025 :",
    rmse_adapter_2025
)

print(
    "MAE brut 2025 :",
    mae_raw_2025
)

print(
    "MAE adaptateur 2025 :",
    mae_adapter_2025
)


print(
    "Gain RMSE (%) :",
    100
    * (
        rmse_raw_2025
        -
        rmse_adapter_2025
    )
    /
    rmse_raw_2025
)

RMSE brut 2025 : 1.6808674646711306
RMSE adaptateur 2025 : 1.2551360596483996
MAE brut 2025 : 1.2997397260273973
MAE adaptateur 2025 : 0.9901194708240748
Gain RMSE (%) : 25.328076958526133


In [ ]:
# ==========================================
# MONTHLY OUT-OF-SAMPLE CHECK
# ==========================================

weather_2025[
    "month"
] = (
    weather_2025[
        "timestamp_utc"
    ]
    .dt
    .month
)


weather_2025[
    "abs_error_raw"
] = np.abs(
    error_raw
)


weather_2025[
    "abs_error_adapter"
] = np.abs(
    error_adapter
)


monthly_adapter_check = (
    weather_2025
    .groupby(
        "month"
    )
    .agg(
        n=(
            "temperature_db",
            "size",
        ),

        mae_raw=(
            "abs_error_raw",
            "mean",
        ),

        mae_adapter=(
            "abs_error_adapter",
            "mean",
        ),
    )
    .reset_index()
)


monthly_adapter_check[
    "improvement_percent"
] = (
    100
    * (
        monthly_adapter_check[
            "mae_raw"
        ]
        -
        monthly_adapter_check[
            "mae_adapter"
        ]
    )
    /
    monthly_adapter_check[
        "mae_raw"
    ]
)


display(
    monthly_adapter_check
)

,month,n,mae_raw,mae_adapter,improvement_percent
0,1,744,0.734597,0.742875,-1.126916
1,2,672,0.962738,0.761126,20.941495
2,3,744,1.239812,0.881757,28.8798
3,4,720,1.44175,1.069542,25.816415
4,5,744,1.582688,1.192374,24.661457
5,6,720,1.785556,1.446795,18.972269
6,7,744,1.660376,1.263445,23.906092
7,8,744,1.918548,1.422162,25.873019
8,9,720,1.307667,0.847700,35.174633
9,10,744,1.11836,0.775392,30.667066


### Partie C — Analyse et améliorations guidées par la théorie

La forte corrélation entre les deux sources météorologiques ne garantit
pas leur accord numérique.

La calibration affine globale testée sur 2025 ne réduit le RMSE que
d'environ 1 %, ce qui montre que la différence entre les sources n'est
pas décrite correctement par un simple changement de niveau et d'échelle.

Un adaptateur légèrement plus riche est donc testé avec des variables
cycliques représentant l'heure et le mois.

Pour éviter toute fuite d'information, cet adaptateur est entraîné sur
2024 puis évalué sur 2025.

L'année 2026 reste totalement exclue de l'estimation et de la sélection
de cet adaptateur.

RMSE brut 2025 :
RMSE adaptateur 2025 :
MAE brut 2025 :
MAE adaptateur 2025 :
Gain RMSE (%) :

In [ ]:
# ==========================================
# FREEZE WEATHER SOURCE ADAPTER
# ==========================================

from pathlib import Path
import joblib


PROJECT_DIR = Path(
    r"C:\Users\thiba\OneDrive\Bureau\Projet energie"
)

MODELS_DIR = (
    PROJECT_DIR
    / "models"
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


WEATHER_ADAPTER_PATH = (
    MODELS_DIR
    / "final_weather_source_adapter.joblib"
)


weather_adapter_state = {

    "model":
        weather_source_adapter,

    "features":
        WEATHER_FEATURES,

    "training_period":
        "2024",

    "validation_period":
        "2025",

    "rmse_raw_2025":
        float(
            rmse_raw_2025
        ),

    "rmse_adapter_2025":
        float(
            rmse_adapter_2025
        ),

    "mae_raw_2025":
        float(
            mae_raw_2025
        ),

    "mae_adapter_2025":
        float(
            mae_adapter_2025
        ),

    "rmse_improvement_percent":
        float(
            100
            * (
                rmse_raw_2025
                - rmse_adapter_2025
            )
            / rmse_raw_2025
        ),

    "uses_2026_for_fit":
        False,
}


joblib.dump(
    weather_adapter_state,
    WEATHER_ADAPTER_PATH,
)


print(
    "Adaptateur météo figé :",
    WEATHER_ADAPTER_PATH
)

print(
    "Fichier existe :",
    WEATHER_ADAPTER_PATH.exists()
)

Adaptateur météo figé : C:\Users\thiba\OneDrive\Bureau\Projet energie\models\final_weather_source_adapter.joblib
Fichier existe : True


**Conclusion :** l'adaptateur de source météorologique est entraîné
exclusivement sur 2024 et validé sur 2025. Il réduit le RMSE météorologique
hors échantillon d'environ 25,3 %. Il est désormais figé et ne sera plus
modifié pendant l'évaluation 2026.

In [ ]:
# ==========================================
# FINAL 2026 WEATHER — FAST VERSION
# ==========================================

FINAL_2026_CUTOFF = pd.Timestamp(
    "2026-09-26 23:00:00+00:00"
)


# Réutilisation de la météo déjà téléchargée
france_new_2026 = (
    weather_long[
        (
            weather_long["city"] == "France"
        )
        &
        (
            weather_long["timestamp_utc"]
            >= pd.Timestamp(
                "2026-01-01 00:00:00+00:00"
            )
        )
        &
        (
            weather_long["timestamp_utc"]
            <= FINAL_2026_CUTOFF
        )
    ]
    [
        [
            "timestamp_utc",
            "temperature",
        ]
    ]
    .rename(
        columns={
            "temperature":
                "temperature_new"
        }
    )
    .copy()
    .sort_values(
        "timestamp_utc"
    )
    .reset_index(
        drop=True
    )
)


# Variables nécessaires à l'adaptateur
weather_2026_adapter = (
    add_weather_adapter_features(
        france_new_2026
    )
)


# Application de l'adaptateur figé
weather_2026_adapter[
    "temperature"
] = (
    weather_source_adapter.predict(
        weather_2026_adapter[
            WEATHER_FEATURES
        ]
    )
)


weather_final_2026 = (
    weather_2026_adapter[
        [
            "timestamp_utc",
            "temperature",
        ]
    ]
    .copy()
)


weather_final_2026[
    "city"
] = "France"


weather_final_2026 = (
    weather_final_2026[
        [
            "timestamp_utc",
            "city",
            "temperature",
        ]
    ]
)


print(
    "Observations météo :",
    len(weather_final_2026)
)

print(
    "Début :",
    weather_final_2026[
        "timestamp_utc"
    ].min()
)

print(
    "Fin :",
    weather_final_2026[
        "timestamp_utc"
    ].max()
)

print(
    "NaN température :",
    weather_final_2026[
        "temperature"
    ].isna().sum()
)


display(
    weather_final_2026.head()
)

display(
    weather_final_2026.tail()
)

,timestamp_utc,city,temperature
0,2026-01-01 00:00:00+00:00,France,0.059273
1,2026-01-01 01:00:00+00:00,France,-0.211674
2,2026-01-01 02:00:00+00:00,France,-0.513620
3,2026-01-01 03:00:00+00:00,France,-0.751294
4,2026-01-01 04:00:00+00:00,France,-1.140928


,timestamp_utc,city,temperature
6451,2026-09-26 19:00:00+00:00,France,20.381556
6452,2026-09-26 20:00:00+00:00,France,19.816547
6453,2026-09-26 21:00:00+00:00,France,18.945670
6454,2026-09-26 22:00:00+00:00,France,18.265193
6455,2026-09-26 23:00:00+00:00,France,17.524407


Observations météo : 6456
Début : 2026-01-01 00:00:00+00:00
Fin : 2026-09-26 23:00:00+00:00
NaN température : 0


In [ ]:
# ==========================================
# FINAL 2026 CONSUMPTION
# ==========================================

consumption_final_2026 = (
    consumption_complete_2026[
        consumption_complete_2026[
            "timestamp_utc"
        ]
        <= FINAL_2026_CUTOFF
    ]
    [
        [
            "timestamp_utc",
            "consumption_mw",
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    "Observations consommation :",
    len(
        consumption_final_2026
    )
)

print(
    "Début :",
    consumption_final_2026[
        "timestamp_utc"
    ].min()
)

print(
    "Fin :",
    consumption_final_2026[
        "timestamp_utc"
    ].max()
)

Observations consommation : 12911
Début : 2026-01-01 00:00:00+00:00
Fin : 2026-09-26 23:00:00+00:00


In [ ]:
# ==========================================
# CREATE FINAL EVALUATION TABLES
# ==========================================

with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            CREATE TABLE IF NOT EXISTS
            energy.consumption_2026_eval (

                timestamp_utc
                    TIMESTAMPTZ PRIMARY KEY,

                consumption_mw
                    DOUBLE PRECISION NOT NULL

            );
        """)


        cur.execute("""
            CREATE TABLE IF NOT EXISTS
            energy.weather_2026_eval (

                timestamp_utc
                    TIMESTAMPTZ NOT NULL,

                city
                    VARCHAR NOT NULL,

                temperature
                    DOUBLE PRECISION NOT NULL,

                PRIMARY KEY (
                    timestamp_utc,
                    city
                )

            );
        """)


        cur.execute(
            "TRUNCATE TABLE "
            "energy.consumption_2026_eval;"
        )

        cur.execute(
            "TRUNCATE TABLE "
            "energy.weather_2026_eval;"
        )


    conn.commit()


print(
    "Tables d'évaluation créées."
)

Tables d'évaluation créées.


In [ ]:
# ==========================================
# INSERT FINAL 2026 DATA
# ==========================================

with get_connection() as conn:

    with conn.cursor() as cur:

        cur.executemany(
            """
            INSERT INTO
            energy.consumption_2026_eval (
                timestamp_utc,
                consumption_mw
            )

            VALUES (%s, %s);
            """,

            list(
                consumption_final_2026[
                    [
                        "timestamp_utc",
                        "consumption_mw",
                    ]
                ]
                .itertuples(
                    index=False,
                    name=None,
                )
            ),
        )


        cur.executemany(
            """
            INSERT INTO
            energy.weather_2026_eval (
                timestamp_utc,
                city,
                temperature
            )

            VALUES (%s, %s, %s);
            """,

            list(
                weather_final_2026[
                    [
                        "timestamp_utc",
                        "city",
                        "temperature",
                    ]
                ]
                .itertuples(
                    index=False,
                    name=None,
                )
            ),
        )


    conn.commit()


print(
    "Données 2026 insérées."
)

Données 2026 insérées.


In [ ]:
# ==========================================
# HISTORICAL + FINAL 2026 SOURCES
# ==========================================

with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            CREATE OR REPLACE VIEW
            energy.consumption_eval_source
            AS

            SELECT
                timestamp_utc,
                consumption_mw

            FROM energy.consumption

            WHERE timestamp_utc <
                '2026-01-01 00:00:00+00'

            UNION ALL

            SELECT
                timestamp_utc,
                consumption_mw

            FROM energy.consumption_2026_eval;
        """)


        cur.execute("""
            CREATE OR REPLACE VIEW
            energy.weather_eval_source
            AS

            SELECT
                timestamp_utc,
                city,
                temperature

            FROM energy.weather

            WHERE timestamp_utc <
                '2026-01-01 00:00:00+00'

            UNION ALL

            SELECT
                timestamp_utc,
                city,
                temperature

            FROM energy.weather_2026_eval;
        """)


    conn.commit()


print(
    "Sources combinées créées."
)

Sources combinées créées.


In [ ]:
# ==========================================
# OPTIMIZED FINAL 2026 FEATURE PIPELINE
# ==========================================

WARMUP_START = (
    "2025-12-24 00:00:00+00"
)

FINAL_CUTOFF = (
    "2026-09-26 23:00:00+00"
)


eval_features_sql = (
    ml_features_definition

    .replace(
        "energy.consumption",
        "energy.consumption_eval_source",
    )

    .replace(
        "energy.weather",
        "energy.weather_eval_source",
    )

    .replace(
        (
            "WHERE c.timestamp_utc < "
            "'2026-01-01 00:00:00+00'"
            "::timestamp with time zone"
        ),
        (
            "WHERE c.timestamp_utc >= "
            f"'{WARMUP_START}'"
            "::timestamp with time zone "
            "AND c.timestamp_utc <= "
            f"'{FINAL_CUTOFF}'"
            "::timestamp with time zone"
        ),
    )
)


eval_features_sql = (
    eval_features_sql
    .strip()
    .rstrip(";")
)


assert (
    "energy.consumption_eval_source"
    in eval_features_sql
)

assert (
    "energy.weather_eval_source"
    in eval_features_sql
)

assert (
    WARMUP_START
    in eval_features_sql
)

assert (
    FINAL_CUTOFF
    in eval_features_sql
)


print(
    "SQL optimisé : OK"
)

SQL optimisé : OK


In [ ]:
assert (
    "energy.consumption_eval_source"
    in eval_features_sql
)

assert (
    "energy.weather_eval_source"
    in eval_features_sql
)

assert (
    "2026-09-26 23:00:00+00"
    in eval_features_sql
)


print(
    "SQL d'évaluation : OK"
)

SQL d'évaluation : OK


In [ ]:
# ==========================================
# FAST FINAL 2026 FEATURE PIPELINE
# ==========================================

import numpy as np
import pandas as pd


# ------------------------------------------
# 1. Sources déjà calculées
# ------------------------------------------

cons = (
    consumption_final_2026[
        [
            "timestamp_utc",
            "consumption_mw",
        ]
    ]
    .copy()
    .sort_values("timestamp_utc")
    .reset_index(drop=True)
)


weather = (
    weather_final_2026[
        [
            "timestamp_utc",
            "temperature",
        ]
    ]
    .copy()
    .sort_values("timestamp_utc")
    .reset_index(drop=True)
)


# ------------------------------------------
# 2. Météo horaire -> consommation 30 min
# équivalent au JOIN LATERAL historique
# ------------------------------------------

df = pd.merge_asof(
    cons,
    weather,
    on="timestamp_utc",
    direction="backward",
)


# ------------------------------------------
# 3. Lags consommation
# ------------------------------------------

df["consumption_lag_30m"] = (
    df["consumption_mw"].shift(1)
)

df["consumption_lag_1h"] = (
    df["consumption_mw"].shift(2)
)

df["consumption_lag_24h"] = (
    df["consumption_mw"].shift(48)
)

df["consumption_lag_7d"] = (
    df["consumption_mw"].shift(336)
)


# ------------------------------------------
# 4. Rolling consommation
# uniquement données passées
# ------------------------------------------

past_consumption = (
    df["consumption_mw"].shift(1)
)

df["consumption_mean_3h"] = (
    past_consumption
    .rolling(6)
    .mean()
)

df["consumption_mean_24h"] = (
    past_consumption
    .rolling(48)
    .mean()
)

df["consumption_mean_7d"] = (
    past_consumption
    .rolling(336)
    .mean()
)

df["consumption_std_24h"] = (
    past_consumption
    .rolling(48)
    .std()
)


# ------------------------------------------
# 5. Features température
# ------------------------------------------

df["temp_lag_1h"] = (
    df["temperature"].shift(2)
)

df["temp_lag_24h"] = (
    df["temperature"].shift(48)
)

df["temp_lag_7d"] = (
    df["temperature"].shift(336)
)

df["temp_mean_24h"] = (
    df["temperature"]
    .shift(1)
    .rolling(48)
    .mean()
)


# ------------------------------------------
# 6. Heure locale France
# ------------------------------------------

local_time = (
    df["timestamp_utc"]
    .dt
    .tz_convert("Europe/Paris")
)


df["local_hour"] = (
    local_time.dt.hour
    +
    local_time.dt.minute / 60
)


# PostgreSQL EXTRACT(DOW):
# dimanche = 0 ... samedi = 6

df["day_of_week"] = (
    (local_time.dt.dayofweek + 1) % 7
)

df["month"] = (
    local_time.dt.month
)

df["is_weekend"] = (
    df["day_of_week"]
    .isin([0, 6])
    .astype(int)
)


# ------------------------------------------
# 7. Encodage cyclique
# ------------------------------------------

df["hour_sin"] = np.sin(
    2
    * np.pi
    * df["local_hour"]
    / 24
)

df["hour_cos"] = np.cos(
    2
    * np.pi
    * df["local_hour"]
    / 24
)

df["month_sin"] = np.sin(
    2
    * np.pi
    * df["month"]
    / 12
)

df["month_cos"] = np.cos(
    2
    * np.pi
    * df["month"]
    / 12
)


# ------------------------------------------
# 8. Ordre exact des 23 colonnes
# ------------------------------------------

FINAL_COLUMNS = [
    "timestamp_utc",
    "consumption_mw",
    "consumption_lag_30m",
    "consumption_lag_1h",
    "consumption_lag_24h",
    "consumption_lag_7d",
    "consumption_mean_3h",
    "consumption_mean_24h",
    "consumption_mean_7d",
    "consumption_std_24h",
    "temperature",
    "temp_lag_1h",
    "temp_lag_24h",
    "temp_lag_7d",
    "temp_mean_24h",
    "local_hour",
    "day_of_week",
    "month",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
]


final_2026_df = (
    df[
        df["consumption_lag_7d"]
        .notna()
    ][
        FINAL_COLUMNS
    ]
    .reset_index(drop=True)
)


print(
    "Shape final 2026 :",
    final_2026_df.shape
)

print(
    "Début :",
    final_2026_df["timestamp_utc"].min()
)

print(
    "Fin :",
    final_2026_df["timestamp_utc"].max()
)

print(
    "NaN totaux :",
    final_2026_df.isna().sum().sum()
)


display(
    final_2026_df.head()
)

Shape final 2026 : (12575, 23)
Début : 2026-01-08 00:00:00+00:00
Fin : 2026-09-26 23:00:00+00:00
NaN totaux : 0


,timestamp_utc,consumption_mw,consumption_lag_30m,consumption_lag_1h,consumption_lag_24h,consumption_lag_7d,consumption_mean_3h,consumption_mean_24h,consumption_mean_7d,consumption_std_24h,...,temp_lag_7d,temp_mean_24h,local_hour,day_of_week,month,is_weekend,hour_sin,hour_cos,month_sin,month_cos
0,2026-01-08 00:00:00+00:00,71847.0,74055.0,75081.0,77582.0,69489.0,75747.166667,81426.312500,74948.157738,5103.759720,...,0.059273,0.908615,1.0,4,1,0,0.258819,0.965926,0.5,0.866025
1,2026-01-08 00:30:00+00:00,71704.0,71847.0,74055.0,77622.0,69379.0,75004.666667,81306.833333,74955.175595,5260.394675,...,0.059273,0.991618,1.5,4,1,0,0.382683,0.923880,0.5,0.866025
2,2026-01-08 01:00:00+00:00,71042.0,71704.0,71847.0,77253.0,69237.0,74235.333333,81183.541667,74962.095238,5415.657210,...,-0.211674,1.074622,2.0,4,1,0,0.500000,0.866025,0.5,0.866025
3,2026-01-08 01:30:00+00:00,70196.0,71042.0,71704.0,76783.0,69074.0,73236.000000,81054.145833,74967.467262,5583.176008,...,-0.211674,1.159155,2.5,4,1,0,0.608761,0.793353,0.5,0.866025
4,2026-01-08 02:00:00+00:00,68755.0,70196.0,71042.0,75485.0,67831.0,72320.833333,80916.916667,74970.806548,5768.273221,...,-0.513620,1.243689,3.0,4,1,0,0.707107,0.707107,0.5,0.866025


In [ ]:
# ==========================================
# FINAL 2026 DATASET AUDIT
# ==========================================

print(
    "Shape :",
    final_2026_df.shape
)

print(
    "Début :",
    final_2026_df[
        "timestamp_utc"
    ].min()
)

print(
    "Fin :",
    final_2026_df[
        "timestamp_utc"
    ].max()
)

print(
    "NaN totaux :",
    final_2026_df
    .isna()
    .sum()
    .sum()
)

print(
    "Colonnes :",
    len(
        final_2026_df.columns
    )
)

Shape : (12575, 23)
Début : 2026-01-08 00:00:00+00:00
Fin : 2026-09-26 23:00:00+00:00
NaN totaux : 0
Colonnes : 23


In [ ]:
# ==========================================
# INSPECT FINAL BLOCK 7 STATE
# ==========================================

import joblib


BLOCK7_PATH = (
    MODELS_DIR
    / "block7_final_state.joblib"
)


block7_state = joblib.load(
    BLOCK7_PATH
)


print(
    "Type :",
    type(
        block7_state
    )
)

if isinstance(
    block7_state,
    dict
):

    print(
        "\nClés disponibles :"
    )

    for key in block7_state.keys():

        print(
            "-",
            key,
            ":",
            type(
                block7_state[
                    key
                ]
            ).__name__
        )

Type : <class 'dict'>

Clés disponibles :
- model_name : str
- index_2025 : DatetimeIndex
- y_2025 : ndarray
- prediction_2025 : ndarray
- metrics_2025 : dict
- rmse_2025 : float64
- event_columns : list
- event_lambda : float
- ewma_selected : bool
- ewma_alpha : float
- dst_expert_selected : bool
- dm_summary : DataFrame


In [ ]:
# ==========================================
# REUSE FINAL WEATHER IF ALREADY AVAILABLE
# ==========================================

if (
    "weather_final_2026" in globals()
    and not weather_final_2026.empty
):

    print("Météo 2026 déjà disponible : réutilisation.")

    print(
        "Shape :",
        weather_final_2026.shape
    )

    print(
        "Début :",
        weather_final_2026[
            "timestamp_utc"
        ].min()
    )

    print(
        "Fin :",
        weather_final_2026[
            "timestamp_utc"
        ].max()
    )

else:

    print(
        "weather_final_2026 absent."
    )

Météo 2026 déjà disponible : réutilisation.
Shape : (6456, 3)
Début : 2026-01-01 00:00:00+00:00
Fin : 2026-09-26 23:00:00+00:00


In [ ]:
# ==========================================
# FINAL WEATHER AUDIT
# ==========================================

weather_steps = (
    weather_final_2026[
        "timestamp_utc"
    ]
    .diff()
    .dropna()
)


print(
    "Observations météo :",
    len(
        weather_final_2026
    )
)

print(
    "Début météo :",
    weather_final_2026[
        "timestamp_utc"
    ].min()
)

print(
    "Fin météo :",
    weather_final_2026[
        "timestamp_utc"
    ].max()
)

print(
    "NaN température :",
    weather_final_2026[
        "temperature"
    ].isna().sum()
)

print(
    "Doublons :",
    weather_final_2026[
        "timestamp_utc"
    ].duplicated().sum()
)

print(
    "Part pas 1h :",
    (
        weather_steps
        ==
        pd.Timedelta(
            hours=1
        )
    ).mean()
)

Observations météo : 6456
Début météo : 2026-01-01 00:00:00+00:00
Fin météo : 2026-09-26 23:00:00+00:00
NaN température : 0
Doublons : 0
Part pas 1h : 1.0


In [ ]:
# ==========================================
# FINAL 2026 WEATHER — FAST / NO DOWNLOAD
# ==========================================

FINAL_2026_CUTOFF = pd.Timestamp(
    "2026-09-26 23:00:00+00:00"
)


france_new_2026 = (
    weather_long[
        (weather_long["city"] == "France")
        &
        (
            weather_long["timestamp_utc"]
            >= pd.Timestamp(
                "2026-01-01 00:00:00+00:00"
            )
        )
        &
        (
            weather_long["timestamp_utc"]
            <= FINAL_2026_CUTOFF
        )
    ]
    [
        [
            "timestamp_utc",
            "temperature",
        ]
    ]
    .rename(
        columns={
            "temperature":
                "temperature_new"
        }
    )
    .copy()
)


weather_2026_adapter = (
    add_weather_adapter_features(
        france_new_2026
    )
)


weather_2026_adapter[
    "temperature"
] = (
    weather_source_adapter.predict(
        weather_2026_adapter[
            WEATHER_FEATURES
        ]
    )
)


weather_final_2026 = (
    weather_2026_adapter[
        [
            "timestamp_utc",
            "temperature",
        ]
    ]
    .copy()
)


weather_final_2026[
    "city"
] = "France"


weather_final_2026 = (
    weather_final_2026[
        [
            "timestamp_utc",
            "city",
            "temperature",
        ]
    ]
    .sort_values(
        "timestamp_utc"
    )
    .reset_index(
        drop=True
    )
)


print(
    "Météo 2026 prête :",
    weather_final_2026.shape
)

Météo 2026 prête : (6456, 3)


In [ ]:
# ==========================================
# FINAL 2026 CONSUMPTION
# ==========================================

consumption_final_2026 = (
    consumption_complete_2026[
        consumption_complete_2026[
            "timestamp_utc"
        ]
        <= FINAL_2026_CUTOFF
    ]
    [
        [
            "timestamp_utc",
            "consumption_mw",
        ]
    ]
    .copy()
    .sort_values(
        "timestamp_utc"
    )
    .reset_index(
        drop=True
    )
)


display(
    consumption_final_2026.head()
)

display(
    consumption_final_2026.tail()
)

,timestamp_utc,consumption_mw
0,2026-01-01 00:00:00+00:00,69489.0
1,2026-01-01 00:30:00+00:00,69379.0
2,2026-01-01 01:00:00+00:00,69237.0
3,2026-01-01 01:30:00+00:00,69074.0
4,2026-01-01 02:00:00+00:00,67831.0


,timestamp_utc,consumption_mw
12906,2026-09-26 21:00:00+00:00,39915.0
12907,2026-09-26 21:30:00+00:00,39484.0
12908,2026-09-26 22:00:00+00:00,39185.0
12909,2026-09-26 22:30:00+00:00,38006.0
12910,2026-09-26 23:00:00+00:00,36457.0


In [ ]:
# ==========================================
# FINAL CONSUMPTION AUDIT
# ==========================================

consumption_steps = (
    consumption_final_2026[
        "timestamp_utc"
    ]
    .diff()
    .dropna()
)


print(
    "Observations consommation :",
    len(
        consumption_final_2026
    )
)

print(
    "Début consommation :",
    consumption_final_2026[
        "timestamp_utc"
    ].min()
)

print(
    "Fin consommation :",
    consumption_final_2026[
        "timestamp_utc"
    ].max()
)

print(
    "NaN consommation :",
    consumption_final_2026[
        "consumption_mw"
    ].isna().sum()
)

print(
    "Doublons :",
    consumption_final_2026[
        "timestamp_utc"
    ].duplicated().sum()
)

print(
    "Part pas 30 min :",
    (
        consumption_steps
        ==
        pd.Timedelta(
            minutes=30
        )
    ).mean()
)

Observations consommation : 12911
Début consommation : 2026-01-01 00:00:00+00:00
Fin consommation : 2026-09-26 23:00:00+00:00
NaN consommation : 0
Doublons : 0
Part pas 30 min : 1.0


In [ ]:
# ==========================================
# FINAL SOURCES VALIDATION
# ==========================================

assert (
    weather_final_2026[
        "temperature"
    ].isna().sum()
    == 0
)

assert (
    consumption_final_2026[
        "consumption_mw"
    ].isna().sum()
    == 0
)

assert (
    weather_final_2026[
        "timestamp_utc"
    ].duplicated().sum()
    == 0
)

assert (
    consumption_final_2026[
        "timestamp_utc"
    ].duplicated().sum()
    == 0
)

assert (
    weather_final_2026[
        "timestamp_utc"
    ].max()
    ==
    FINAL_2026_CUTOFF
)

assert (
    consumption_final_2026[
        "timestamp_utc"
    ].max()
    ==
    FINAL_2026_CUTOFF
)


print(
    "Validation finale des sources 2026 : OK"
)

Validation finale des sources 2026 : OK


**Conclusion :** les deux sources utilisées pour l'évaluation finale
sont maintenant figées au 26 septembre 2026 à 23:00 UTC.

La consommation conserve un pas de 30 minutes et la météo un pas horaire.
Aucune observation 2026 n'a été utilisée pour réentraîner ou sélectionner
le modèle de consommation.

In [ ]:
# ==========================================
# CHECKPOINT FINAL 2026 DATASET
# RAPIDE
# ==========================================

FINAL_2026_PATH = (
    PROJECT_DIR
    / "data_processed"
    / "final_2026_dataset.joblib"
)


joblib.dump(
    final_2026_df,
    FINAL_2026_PATH,
)


print(
    "Dataset 2026 sauvegardé :",
    FINAL_2026_PATH
)

print(
    "Shape :",
    final_2026_df.shape
)

print(
    "NaN :",
    final_2026_df.isna().sum().sum()
)

Dataset 2026 sauvegardé : C:\Users\thiba\OneDrive\Bureau\Projet energie\data_processed\final_2026_dataset.joblib
Shape : (12575, 23)
NaN : 0


In [ ]:
# ==========================================
# EXTRACT BLOCK 3 FEATURE BUILDER — RAPIDE
# ==========================================

import json
from pathlib import Path


PROJECT_DIR = Path(
    r"C:\Users\thiba\OneDrive\Bureau\Projet energie"
)

OUTPUT_PATH = (
    PROJECT_DIR
    / "data_processed"
    / "block3_feature_builder.txt"
)


patterns = [
    "X_block3_weather_nonlinear.joblib",
    "y_block3_weather.joblib",
    "weather_nonlinear",
]


parts = []
seen = set()


for path in PROJECT_DIR.glob("*.ipynb"):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        nb = json.load(f)

    cells = nb["cells"]

    for i, cell in enumerate(cells):

        source = "".join(
            cell.get("source", [])
        )

        if not any(
            p.lower() in source.lower()
            for p in patterns
        ):
            continue

        # On prend suffisamment de contexte avant
        # la sauvegarde pour récupérer les formules.
        for j in range(
            max(0, i - 15),
            min(len(cells), i + 4),
        ):

            key = (path.name, j)

            if key in seen:
                continue

            seen.add(key)

            src = "".join(
                cells[j].get("source", [])
            )

            parts.append(
                "\n"
                + "=" * 80
                + "\n"
                + f"FILE: {path.name}\n"
                + f"CELL: {j}\n"
                + "=" * 80
                + "\n"
                + src
            )


OUTPUT_PATH.write_text(
    "\n".join(parts),
    encoding="utf-8",
)


print("Créé :", OUTPUT_PATH)

print(
    "Taille :",
    round(
        OUTPUT_PATH.stat().st_size / 1024,
        1
    ),
    "Ko"
)

Créé : C:\Users\thiba\OneDrive\Bureau\Projet energie\data_processed\block3_feature_builder.txt
Taille : 48.5 Ko


In [3]:
# ==========================================
# EXTRACT X_BLOCK1 FINAL BUILDER — RAPIDE
# ==========================================

import json


path = (
    PROJECT_DIR
    / "01_high_dimensional_linear_modeling.ipynb"
)

output = (
    PROJECT_DIR
    / "data_processed"
    / "block1_final_builder.txt"
)


with open(
    path,
    "r",
    encoding="utf-8",
) as f:

    nb = json.load(f)


cells = nb["cells"]

matches = []


for i, cell in enumerate(cells):

    source = "".join(
        cell.get(
            "source",
            []
        )
    )

    if (
        "X_block1_final.joblib"
        in source
        or
        "y_block1_final.joblib"
        in source
    ):

        matches.append(i)


print(
    "Cellules trouvées :",
    matches
)


parts = []


for i in matches:

    # On récupère assez de contexte avant la sauvegarde
    for j in range(
        max(0, i - 25),
        min(len(cells), i + 4),
    ):

        source = "".join(
            cells[j].get(
                "source",
                []
            )
        )

        parts.append(
            "\n"
            + "=" * 75
            + "\n"
            + f"CELL {j}\n"
            + "=" * 75
            + "\n"
            + source
        )


output.write_text(
    "\n".join(parts),
    encoding="utf-8",
)


print(
    "Créé :",
    output
)

print(
    "Taille :",
    round(
        output.stat().st_size
        / 1024,
        1,
    ),
    "Ko"
)

Cellules trouvées : [475, 476, 477]
Créé : C:\Users\thiba\OneDrive\Bureau\Projet energie\data_processed\block1_final_builder.txt
Taille : 71.2 Ko


In [18]:
# ============================================================
# ÉVALUATION FINALE 2026
# RECONSTRUCTION CANONIQUE DES 429 FEATURES
#
# Bloc 1 : 416 features
# + météo : 7 features
# + météo non linéaire : 6 features
# = 429 features
# ============================================================

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from src.database.postgres import get_connection


# ============================================================
# 1. CHEMINS
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\thiba\OneDrive\Bureau\Projet energie"
)

DATA_DIR = (
    PROJECT_DIR
    / "data_processed"
)

MODELS_DIR = (
    PROJECT_DIR
    / "models"
)


# ============================================================
# 2. DATASET FINAL 2026
# ============================================================

FINAL_DATA_PATH = (
    DATA_DIR
    / "final_2026_dataset.joblib"
)

final_2026_df = joblib.load(
    FINAL_DATA_PATH
).copy()


final_2026_df[
    "timestamp_utc"
] = pd.to_datetime(
    final_2026_df[
        "timestamp_utc"
    ],
    utc=True
)


final_2026_df = (
    final_2026_df
    .set_index(
        "timestamp_utc"
    )
    .sort_index()
)


FINAL_INDEX = (
    final_2026_df.index
)

FINAL_START = (
    FINAL_INDEX.min()
)

FINAL_CUTOFF = (
    FINAL_INDEX.max()
)


print(
    "Période finale :",
    FINAL_START,
    "->",
    FINAL_CUTOFF
)

print(
    "Observations finales :",
    len(FINAL_INDEX)
)


# ============================================================
# 3. SCHÉMA EXACT ATTENDU PAR LE MODÈLE DU BLOC 3
# ============================================================

RIDGE_PATH = (
    MODELS_DIR
    / "block3_conformal_ridge.joblib"
)

conformal_ridge = joblib.load(
    RIDGE_PATH
)


if not hasattr(
    conformal_ridge,
    "feature_names_in_"
):
    raise RuntimeError(
        "Le Ridge ne contient pas feature_names_in_."
    )


EXPECTED_FEATURES = list(
    conformal_ridge.feature_names_in_
)


print(
    "\nFeatures attendues par Ridge :",
    len(EXPECTED_FEATURES)
)


assert (
    len(EXPECTED_FEATURES)
    == 429
), (
    "Le modèle final devrait attendre "
    "429 features."
)


# ============================================================
# 4. CHARGEMENT HISTORIQUE + 2026
#
# On a besoin d'un an d'historique environ
# pour les lags saisonniers du Bloc 1.
# ============================================================

HISTORY_START = pd.Timestamp(
    "2025-01-01 00:00:00+00:00"
)


with get_connection() as conn:

    consumption_all = pd.read_sql(
        """
        SELECT
            timestamp_utc,
            consumption_mw
        FROM energy.consumption_eval_source
        WHERE timestamp_utc >= %s
          AND timestamp_utc <= %s
        ORDER BY timestamp_utc;
        """,
        conn,
        params=(
            HISTORY_START,
            FINAL_CUTOFF,
        ),
    )


with get_connection() as conn:

    weather_all = pd.read_sql(
        """
        SELECT
            timestamp_utc,
            temperature
        FROM energy.weather_eval_source
        WHERE city = 'France'
          AND timestamp_utc >= %s
          AND timestamp_utc <= %s
        ORDER BY timestamp_utc;
        """,
        conn,
        params=(
            pd.Timestamp(
                "2025-12-20 00:00:00+00:00"
            ),
            FINAL_CUTOFF,
        ),
    )


# ============================================================
# 5. NORMALISATION DES INDEX
# ============================================================

consumption_all[
    "timestamp_utc"
] = pd.to_datetime(
    consumption_all[
        "timestamp_utc"
    ],
    utc=True
)


weather_all[
    "timestamp_utc"
] = pd.to_datetime(
    weather_all[
        "timestamp_utc"
    ],
    utc=True
)


consumption_all = (
    consumption_all
    .drop_duplicates(
        "timestamp_utc"
    )
    .set_index(
        "timestamp_utc"
    )
    .sort_index()
)


weather_all = (
    weather_all
    .drop_duplicates(
        "timestamp_utc"
    )
    .set_index(
        "timestamp_utc"
    )
    .sort_index()
)


# ============================================================
# 6. CADENCE 30 MIN CONSOMMATION
# ============================================================

full_consumption_index = (
    pd.date_range(
        start=consumption_all.index.min(),
        end=FINAL_CUTOFF,
        freq="30min",
        tz="UTC",
    )
)


consumption_all = (
    consumption_all
    .reindex(
        full_consumption_index
    )
)


missing_consumption = int(
    consumption_all[
        "consumption_mw"
    ]
    .isna()
    .sum()
)


print(
    "\nValeurs consommation manquantes "
    "après reindexation :",
    missing_consumption
)


# Petite interpolation uniquement pour les
# trous historiques déjà passés.
#
# À la date du test 2026, ces observations
# appartiennent déjà au passé.
if missing_consumption > 0:

    consumption_all[
        "consumption_mw"
    ] = (
        consumption_all[
            "consumption_mw"
        ]
        .interpolate(
            method="time",
            limit=4,
            limit_direction="both",
        )
    )


remaining_missing = int(
    consumption_all[
        "consumption_mw"
    ]
    .isna()
    .sum()
)


print(
    "NaN consommation restants :",
    remaining_missing
)


if remaining_missing > 0:
    raise RuntimeError(
        "Des trous importants subsistent "
        "dans la consommation."
    )


# ============================================================
# 7. SÉRIE CONSOMMATION
# ============================================================

y_all = (
    consumption_all[
        "consumption_mw"
    ]
    .astype(float)
)


idx = y_all.index


# ============================================================
# 8. LAGS 1 -> 336
# ============================================================

lag_data = {
    f"lag_{lag}":
        y_all.shift(lag)

    for lag in range(
        1,
        337
    )
}


# ============================================================
# 9. LAGS SAISONNIERS
# ============================================================

seasonal_lags = [
    48 * 14,
    48 * 30,
    48 * 90,
    48 * 180,
    48 * 364,
    48 * 365,
    48 * 366,
]


for lag in seasonal_lags:

    lag_data[
        f"lag_{lag}"
    ] = (
        y_all.shift(lag)
    )


lag_df = pd.DataFrame(
    lag_data,
    index=idx,
)


# ============================================================
# 10. ROLLING FEATURES
# ============================================================

windows = [
    6,
    12,
    24,
    48,
    96,
    336,
]


past_y = (
    y_all.shift(1)
)


rolling_data = {}


for window in windows:

    rolling_data[
        f"rolling_mean_{window}"
    ] = (
        past_y
        .rolling(window)
        .mean()
    )

    rolling_data[
        f"rolling_std_{window}"
    ] = (
        past_y
        .rolling(window)
        .std()
    )


rolling_df = pd.DataFrame(
    rolling_data,
    index=idx,
)


# ============================================================
# 11. CALENDRIER CYCLIQUE
#
# Important :
# le Bloc 1 utilisait l'index UTC directement.
# ============================================================

calendar_cyclic = pd.DataFrame(
    index=idx
)


calendar_cyclic[
    "hour_sin"
] = np.sin(
    2
    * np.pi
    * idx.hour
    / 24
)


calendar_cyclic[
    "hour_cos"
] = np.cos(
    2
    * np.pi
    * idx.hour
    / 24
)


calendar_cyclic[
    "dow_sin"
] = np.sin(
    2
    * np.pi
    * idx.dayofweek
    / 7
)


calendar_cyclic[
    "dow_cos"
] = np.cos(
    2
    * np.pi
    * idx.dayofweek
    / 7
)


calendar_cyclic[
    "month_sin"
] = np.sin(
    2
    * np.pi
    * idx.month
    / 12
)


calendar_cyclic[
    "month_cos"
] = np.cos(
    2
    * np.pi
    * idx.month
    / 12
)


# ============================================================
# 12. BASE HAUTE DIMENSION
# ============================================================

X_hd = pd.concat(
    [
        lag_df,
        rolling_df,
        calendar_cyclic,
    ],
    axis=1,
)


# ============================================================
# 13. FEATURES AVANCÉES
# ============================================================

advanced = pd.DataFrame(
    index=idx
)


advanced[
    "diff_1"
] = (
    X_hd["lag_1"]
    - X_hd["lag_2"]
)


advanced[
    "diff_day"
] = (
    X_hd["lag_1"]
    - X_hd["lag_48"]
)


advanced[
    "diff_week"
] = (
    X_hd["lag_1"]
    - X_hd["lag_336"]
)


advanced[
    "trend_3h"
] = (
    X_hd["lag_1"]
    - X_hd["lag_6"]
)


advanced[
    "trend_6h"
] = (
    X_hd["lag_1"]
    - X_hd["lag_12"]
)


advanced[
    "trend_12h"
] = (
    X_hd["lag_1"]
    - X_hd["lag_24"]
)


advanced[
    "dev_mean_3h"
] = (
    X_hd["lag_1"]
    - X_hd["rolling_mean_6"]
)


advanced[
    "dev_mean_day"
] = (
    X_hd["lag_1"]
    - X_hd["rolling_mean_48"]
)


advanced[
    "dev_mean_week"
] = (
    X_hd["lag_1"]
    - X_hd["rolling_mean_336"]
)


advanced[
    "cv_day"
] = (
    X_hd["rolling_std_48"]
    /
    (
        X_hd["rolling_mean_48"]
        + 1e-6
    )
)


advanced[
    "cv_week"
] = (
    X_hd["rolling_std_336"]
    /
    (
        X_hd["rolling_mean_336"]
        + 1e-6
    )
)


X_advanced = pd.concat(
    [
        X_hd,
        advanced,
    ],
    axis=1,
)


# ============================================================
# 14. CALENDRIER FLEXIBLE / DUMMIES
# ============================================================

calendar_raw = pd.DataFrame(
    {
        "hour":
            idx.hour,

        "dayofweek":
            idx.dayofweek,

        "month":
            idx.month,
    },
    index=idx,
)


calendar_dummies = (
    pd.get_dummies(
        calendar_raw,
        columns=[
            "hour",
            "dayofweek",
            "month",
        ],
        drop_first=True,
        dtype=float,
    )
)


X_block1_full = pd.concat(
    [
        X_advanced,
        calendar_dummies,
    ],
    axis=1,
)


# ============================================================
# 15. INTERACTIONS HEBDOMADAIRES
# ============================================================

X_block1_full[
    "lag336_hour_sin"
] = (
    X_block1_full[
        "lag_336"
    ]
    *
    X_block1_full[
        "hour_sin"
    ]
)


X_block1_full[
    "lag336_hour_cos"
] = (
    X_block1_full[
        "lag_336"
    ]
    *
    X_block1_full[
        "hour_cos"
    ]
)


X_block1_full[
    "lag336_dow_sin"
] = (
    X_block1_full[
        "lag_336"
    ]
    *
    X_block1_full[
        "dow_sin"
    ]
)


X_block1_full[
    "lag336_dow_cos"
] = (
    X_block1_full[
        "lag_336"
    ]
    *
    X_block1_full[
        "dow_cos"
    ]
)


# ============================================================
# 16. IDENTIFICATION DES 416 FEATURES BLOC 1
# depuis le schéma exact du modèle 429
# ============================================================

WEATHER_FEATURES = [
    "temp_lag_1h",
    "temp_lag_24h",
    "temp_lag_7d",
    "temp_mean_24h",
    "temp_mean_7d",
    "heating_degree",
    "cooling_degree",
]


WEATHER_NONLINEAR = [
    "temp_squared",
    "temp_cubic",
    "temp_hour_sin",
    "temp_hour_cos",
    "temp_month_sin",
    "temp_month_cos",
]


BLOCK1_EXPECTED = [
    col
    for col in EXPECTED_FEATURES
    if col
    not in (
        WEATHER_FEATURES
        + WEATHER_NONLINEAR
    )
]


print(
    "\nFeatures Bloc 1 attendues :",
    len(BLOCK1_EXPECTED)
)


missing_block1 = [
    col
    for col in BLOCK1_EXPECTED
    if col
    not in X_block1_full.columns
]


print(
    "Features Bloc 1 manquantes :",
    len(missing_block1)
)


if missing_block1:

    print(
        missing_block1[:30]
    )

    raise RuntimeError(
        "Le schéma Bloc 1 ne correspond "
        "pas au modèle entraîné."
    )


X_block1_full = (
    X_block1_full[
        BLOCK1_EXPECTED
    ]
)


assert (
    X_block1_full.shape[1]
    == 416
)


print(
    "Bloc 1 reconstruit :",
    X_block1_full.shape
)


# ============================================================
# 17. MÉTÉO -> 30 MIN
# ============================================================

temperature_hourly = (
    weather_all[
        "temperature"
    ]
    .astype(float)
)


temp_30 = (
    temperature_hourly
    .resample("30min")
    .ffill()
)


weather_features = pd.DataFrame(
    index=temp_30.index
)


weather_features[
    "temp_lag_1h"
] = (
    temp_30.shift(2)
)


weather_features[
    "temp_lag_24h"
] = (
    temp_30.shift(48)
)


weather_features[
    "temp_lag_7d"
] = (
    temp_30.shift(336)
)


weather_features[
    "temp_mean_24h"
] = (
    temp_30
    .shift(2)
    .rolling(48)
    .mean()
)


weather_features[
    "temp_mean_7d"
] = (
    temp_30
    .shift(2)
    .rolling(336)
    .mean()
)


T_weather = (
    weather_features[
        "temp_lag_1h"
    ]
)


weather_features[
    "heating_degree"
] = (
    15
    - T_weather
).clip(
    lower=0
)


weather_features[
    "cooling_degree"
] = (
    T_weather
    - 22
).clip(
    lower=0
)


# ============================================================
# 18. FUSION BLOC 1 + MÉTÉO
# ============================================================

X_weather_2026 = (
    X_block1_full
    .join(
        weather_features,
        how="inner",
    )
)


# ============================================================
# 19. NON-LINÉARITÉS MÉTÉO
# ============================================================

T = (
    X_weather_2026[
        "temp_lag_1h"
    ]
)


X_weather_2026[
    "temp_squared"
] = (
    T ** 2
)


X_weather_2026[
    "temp_cubic"
] = (
    T ** 3
)


X_weather_2026[
    "temp_hour_sin"
] = (
    T
    *
    X_weather_2026[
        "hour_sin"
    ]
)


X_weather_2026[
    "temp_hour_cos"
] = (
    T
    *
    X_weather_2026[
        "hour_cos"
    ]
)


X_weather_2026[
    "temp_month_sin"
] = (
    T
    *
    X_weather_2026[
        "month_sin"
    ]
)


X_weather_2026[
    "temp_month_cos"
] = (
    T
    *
    X_weather_2026[
        "month_cos"
    ]
)


# ============================================================
# 20. ALIGNEMENT EXACT AVEC LES 429 FEATURES D'ENTRAÎNEMENT
# ============================================================

missing_final = [
    col
    for col in EXPECTED_FEATURES
    if col
    not in X_weather_2026.columns
]


print(
    "\nFeatures finales manquantes :",
    len(missing_final)
)


if missing_final:

    print(
        missing_final
    )

    raise RuntimeError(
        "Impossible d'aligner "
        "les 429 features."
    )


X_2026_429 = (
    X_weather_2026[
        EXPECTED_FEATURES
    ]
    .reindex(
        FINAL_INDEX
    )
)


# ============================================================
# 21. TARGET FINAL
# ============================================================

y_2026_final = (
    final_2026_df[
        "consumption_mw"
    ]
    .astype(float)
    .copy()
)


# ============================================================
# 22. AUDIT NaN
# ============================================================

nan_mask = (
    X_2026_429
    .isna()
    .any(axis=1)
)


print(
    "\nLignes avec au moins un NaN :",
    int(
        nan_mask.sum()
    )
)


if nan_mask.any():

    print(
        "Premières dates concernées :"
    )

    print(
        X_2026_429.index[
            nan_mask
        ][:10]
    )


# On conserve uniquement les lignes
# complètement reconstructibles.
valid_mask = (
    ~nan_mask
    &
    y_2026_final.notna()
)


X_2026_429 = (
    X_2026_429.loc[
        valid_mask
    ]
)


y_2026_final = (
    y_2026_final.loc[
        X_2026_429.index
    ]
)


# ============================================================
# 23. AUDIT FINAL
# ============================================================

print(
    "\n"
    + "=" * 60
)

print(
    "DATASET FINAL POUR LE MODÈLE"
)

print(
    "=" * 60
)


print(
    "X :",
    X_2026_429.shape
)


print(
    "y :",
    y_2026_final.shape
)


print(
    "Début :",
    X_2026_429.index.min()
)


print(
    "Fin :",
    X_2026_429.index.max()
)


print(
    "NaN X :",
    X_2026_429
    .isna()
    .sum()
    .sum()
)


print(
    "NaN y :",
    y_2026_final
    .isna()
    .sum()
)


print(
    "Ordre des colonnes correct :",
    list(
        X_2026_429.columns
    )
    ==
    EXPECTED_FEATURES
)


# ============================================================
# 24. CHECKPOINT
# ============================================================

X_2026_PATH = (
    DATA_DIR
    / "X_final_2026_429.joblib"
)

Y_2026_PATH = (
    DATA_DIR
    / "y_final_2026.joblib"
)


joblib.dump(
    X_2026_429,
    X_2026_PATH,
)


joblib.dump(
    y_2026_final,
    Y_2026_PATH,
)


print(
    "\nSauvegardé :",
    X_2026_PATH
)


print(
    "Sauvegardé :",
    Y_2026_PATH
)

Période finale : 2026-01-08 00:00:00+00:00 -> 2026-09-26 23:00:00+00:00
Observations finales : 12575

Features attendues par Ridge : 429


RuntimeError: Variable POSTGRES_PASSWORD absente.

In [8]:
# ============================================================
# FINAL 2026 — STAGE 1
# Exécution PyTorch dans un processus Python 3.11 externe
# ============================================================

from pathlib import Path
import subprocess
import textwrap
import joblib
import pandas as pd


PROJECT_DIR = Path(
    r"C:\Users\thiba\OneDrive\Bureau\Projet energie"
)

SCRIPT_DIR = PROJECT_DIR / "scripts"
SCRIPT_DIR.mkdir(exist_ok=True)

SCRIPT_PATH = (
    SCRIPT_DIR
    / "final_2026_stage1_external.py"
)


# ============================================================
# 1. SCRIPT PYTHON EXTERNE
# ============================================================

script_code = r'''
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    DataLoader,
    TensorDataset,
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)


# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\thiba\OneDrive\Bureau\Projet energie"
)

DATA_DIR = PROJECT_DIR / "data_processed"
MODELS_DIR = PROJECT_DIR / "models"


print("=" * 70)
print("FINAL 2026 — STAGE 1 EXTERNE")
print("=" * 70)

print("PyTorch :", torch.__version__)
print("CUDA :", torch.cuda.is_available())

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device :", device)

if torch.cuda.is_available():

    print(
        "GPU :",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# DONNÉES 2026
# ============================================================

X_2026 = joblib.load(
    DATA_DIR
    / "X_final_2026_429.joblib"
)

y_2026 = joblib.load(
    DATA_DIR
    / "y_final_2026.joblib"
)


y_true = np.asarray(
    y_2026
)


print(
    "\nX :",
    X_2026.shape
)

print(
    "y :",
    y_true.shape
)


# ============================================================
# RIDGE
# ============================================================

ridge_model = joblib.load(
    MODELS_DIR
    / "block3_conformal_ridge.joblib"
)


assert (
    list(ridge_model.feature_names_in_)
    ==
    list(X_2026.columns)
)


pred_ridge_2026 = (
    ridge_model.predict(
        X_2026
    )
)


# ============================================================
# SCALERS MLP1
# ============================================================

res_x_scaler = joblib.load(
    MODELS_DIR
    / "block6_residual_x_scaler.joblib"
)

res_y_scaler = joblib.load(
    MODELS_DIR
    / "block6_residual_y_scaler.joblib"
)


if hasattr(
    res_x_scaler,
    "feature_names_in_"
):

    assert (
        list(res_x_scaler.feature_names_in_)
        ==
        list(X_2026.columns)
    )


X_mlp1_2026 = (
    res_x_scaler
    .transform(
        X_2026
    )
    .astype(
        np.float32
    )
)


# ============================================================
# ARCHITECTURE MLP1
# ============================================================

class ResidualMLP(nn.Module):

    def __init__(
        self,
        n_features,
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                n_features,
                128,
            ),

            nn.GELU(),

            nn.Dropout(
                0.05
            ),

            nn.Linear(
                128,
                64,
            ),

            nn.GELU(),

            nn.Dropout(
                0.05
            ),

            nn.Linear(
                64,
                1,
            ),
        )

    def forward(
        self,
        x,
    ):

        return self.network(x)


mlp1_model = ResidualMLP(
    X_mlp1_2026.shape[1]
).to(
    device
)


state_dict = torch.load(
    MODELS_DIR
    / "block6_residual_mlp.pt",

    map_location=device,
    weights_only=True,
)


mlp1_model.load_state_dict(
    state_dict
)

mlp1_model.eval()


n_params = sum(
    p.numel()
    for p
    in mlp1_model.parameters()
)


print(
    "\nMLP1 paramètres :",
    n_params
)


assert n_params == 63361


# ============================================================
# INFÉRENCE
# ============================================================

dataset_2026 = TensorDataset(
    torch.from_numpy(
        X_mlp1_2026
    )
)


loader_2026 = DataLoader(
    dataset_2026,

    batch_size=4096,

    shuffle=False,

    num_workers=0,

    pin_memory=(
        device.type == "cuda"
    ),
)


pred_scaled = []


with torch.inference_mode():

    for (xb,) in loader_2026:

        xb = xb.to(
            device,
            non_blocking=True,
        )

        if device.type == "cuda":

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):

                pred = mlp1_model(
                    xb
                )

        else:

            pred = mlp1_model(
                xb
            )

        pred_scaled.append(

            pred
            .float()
            .cpu()
            .numpy()

        )


pred_scaled = np.vstack(
    pred_scaled
)


pred_residual_2026 = (
    res_y_scaler
    .inverse_transform(
        pred_scaled
    )
    .ravel()
)


# ============================================================
# RIDGE + MLP1
# ============================================================

pred_stage1_2026 = (
    pred_ridge_2026
    +
    pred_residual_2026
)


# ============================================================
# MÉTRIQUES
# ============================================================

def metrics(
    y,
    pred
):

    mse = mean_squared_error(
        y,
        pred
    )

    return {

        "MSE":
            mse,

        "RMSE":
            np.sqrt(mse),

        "MAE":
            mean_absolute_error(
                y,
                pred
            ),

        "MAPE_%":
            np.mean(
                np.abs(
                    (y - pred)
                    / y
                )
            ) * 100,

        "R2":
            r2_score(
                y,
                pred
            ),
    }


ridge_metrics = metrics(
    y_true,
    pred_ridge_2026
)

stage1_metrics = metrics(
    y_true,
    pred_stage1_2026
)


comparison = pd.DataFrame(
    [

        {
            "Model":
                "Ridge",

            **ridge_metrics,
        },

        {
            "Model":
                "Ridge + MLP1",

            **stage1_metrics,
        },

    ]
)


print(
    "\n"
    + "=" * 70
)

print(
    "RÉSULTATS 2026"
)

print(
    "=" * 70
)

print(
    comparison.round(4)
)


gain = (

    100

    * (
        ridge_metrics["RMSE"]
        -
        stage1_metrics["RMSE"]
    )

    / ridge_metrics["RMSE"]

)


print(
    "\nGain RMSE MLP1 :",
    round(gain, 3),
    "%"
)


# ============================================================
# RÉSIDUS
# ============================================================

residual_stage1 = (
    y_true
    -
    pred_stage1_2026
)


print(
    "\nRésidu moyen :",
    residual_stage1.mean()
)

print(
    "Résidu std :",
    residual_stage1.std()
)


# ============================================================
# SAUVEGARDE
# ============================================================

OUTPUT_PATH = (
    MODELS_DIR
    / "final_2026_stage1.joblib"
)


joblib.dump(

    {

        "index":
            X_2026.index,

        "y_true":
            y_true,

        "pred_ridge":
            pred_ridge_2026,

        "pred_mlp1_residual":
            pred_residual_2026,

        "pred_stage1":
            pred_stage1_2026,

        "ridge_metrics":
            ridge_metrics,

        "stage1_metrics":
            stage1_metrics,

        "gain_rmse_pct":
            gain,

    },

    OUTPUT_PATH

)


print(
    "\nSauvegardé :",
    OUTPUT_PATH
)
'''


SCRIPT_PATH.write_text(
    textwrap.dedent(
        script_code
    ),
    encoding="utf-8",
)


print(
    "Script créé :",
    SCRIPT_PATH
)


# ============================================================
# 2. TEST PYTORCH DANS LE PROCESSUS EXTERNE
# ============================================================

print(
    "\nTest Python 3.11 externe..."
)


test = subprocess.run(

    [
        "py",
        "-3.11",
        "-c",
        (
            "import torch; "
            "print('PyTorch:', torch.__version__); "
            "print('CUDA:', torch.cuda.is_available()); "
            "print('GPU:', torch.cuda.get_device_name(0) "
            "if torch.cuda.is_available() else 'CPU')"
        ),
    ],

    capture_output=True,
    text=True,
)


print(
    test.stdout
)


if test.returncode != 0:

    print(
        "ERREUR TEST :"
    )

    print(
        test.stderr
    )

    raise RuntimeError(
        "PyTorch externe ne fonctionne pas."
    )


# ============================================================
# 3. LANCEMENT STAGE 1
# ============================================================

print(
    "\nLancement de l'évaluation Stage 1..."
)


result = subprocess.run(

    [
        "py",
        "-3.11",
        str(SCRIPT_PATH),
    ],

    cwd=str(
        PROJECT_DIR
    ),

    capture_output=True,
    text=True,
)


print(
    result.stdout
)


if result.returncode != 0:

    print(
        "\nERREUR :"
    )

    print(
        result.stderr
    )

    raise RuntimeError(
        "Le calcul Stage 1 externe a échoué."
    )


# ============================================================
# 4. RECHARGEMENT DANS LE NOTEBOOK
# ============================================================

stage1_2026 = joblib.load(

    PROJECT_DIR
    / "models"
    / "final_2026_stage1.joblib"

)


comparison = pd.DataFrame(
    [

        {
            "Model":
                "Ridge",

            **stage1_2026[
                "ridge_metrics"
            ],
        },

        {
            "Model":
                "Ridge + MLP1",

            **stage1_2026[
                "stage1_metrics"
            ],
        },

    ]
)


print(
    "\nTABLEAU FINAL :"
)

display(
    comparison.round(4)
)


print(
    "\nGain RMSE :",
    round(
        stage1_2026[
            "gain_rmse_pct"
        ],
        3,
    ),
    "%"
)

Script créé : C:\Users\thiba\OneDrive\Bureau\Projet energie\scripts\final_2026_stage1_external.py

Test Python 3.11 externe...
PyTorch: 2.14.0+cu130
CUDA: True
GPU: NVIDIA GeForce GTX 1650 Ti with Max-Q Design


Lancement de l'évaluation Stage 1...
FINAL 2026 â€” STAGE 1 EXTERNE
PyTorch : 2.14.0+cu130
CUDA : True
Device : cuda
GPU : NVIDIA GeForce GTX 1650 Ti with Max-Q Design

X : (12575, 429)
y : (12575,)

MLP1 paramÃ¨tres : 63361

RÃ‰SULTATS 2026
          Model         MSE      RMSE       MAE  MAPE_%      R2
0         Ridge  80825.9674  284.2991  208.3192  0.4433  0.9991
1  Ridge + MLP1  68626.7840  261.9671  192.3009  0.4088  0.9992

Gain RMSE MLP1 : 7.855 %

RÃ©sidu moyen : 12.129212975318804
RÃ©sidu std : 261.68619788132844

SauvegardÃ© : C:\Users\thiba\OneDrive\Bureau\Projet energie\models\final_2026_stage1.joblib


TABLEAU FINAL :


,Model,MSE,RMSE,MAE,MAPE_%,R2
0,Ridge,80825.9674,284.2991,208.3192,0.4433,0.9991
1,Ridge + MLP1,68626.7840,261.9671,192.3009,0.4088,0.9992



Gain RMSE : 7.855 %


In [9]:
# ============================================================
# FINAL 2026 — PATCH TRANSFORMER
# Exécution externe Python 3.11 + CUDA
# ============================================================

from pathlib import Path
import subprocess
import textwrap
import joblib
import pandas as pd


PROJECT_DIR = Path(
    r"C:\Users\thiba\OneDrive\Bureau\Projet energie"
)

SCRIPT_DIR = PROJECT_DIR / "scripts"
SCRIPT_DIR.mkdir(exist_ok=True)

SCRIPT_PATH = (
    SCRIPT_DIR
    / "final_2026_patch_external.py"
)


# ============================================================
# SCRIPT EXTERNE
# ============================================================

script_code = r'''
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)


# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\thiba\OneDrive\Bureau\Projet energie"
)

MODELS_DIR = PROJECT_DIR / "models"


print("=" * 72)
print("FINAL 2026 — PATCH TRANSFORMER")
print("=" * 72)

print("PyTorch :", torch.__version__)
print("CUDA :", torch.cuda.is_available())

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device :", device)

if torch.cuda.is_available():

    print(
        "GPU :",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# 1. CHARGEMENT STAGE 1
# ============================================================

stage1 = joblib.load(
    MODELS_DIR
    / "final_2026_stage1.joblib"
)


index_2026 = pd.DatetimeIndex(
    stage1["index"]
)

y_true = np.asarray(
    stage1["y_true"],
    dtype=float
)

pred_ridge = np.asarray(
    stage1["pred_ridge"],
    dtype=float
)

pred_stage1 = np.asarray(
    stage1["pred_stage1"],
    dtype=float
)


assert (
    len(index_2026)
    ==
    len(y_true)
    ==
    len(pred_stage1)
)


print(
    "\nStage 1 :",
    len(index_2026),
    "observations"
)

print(
    "Début :",
    index_2026[0]
)

print(
    "Fin :",
    index_2026[-1]
)


# ============================================================
# 2. RÉSIDU APRÈS RIDGE + MLP1
# ============================================================

r2_2026 = (
    y_true
    -
    pred_stage1
)


r2_series = pd.Series(
    r2_2026,
    index=index_2026,
    name="r2"
)


print(
    "\nRésidu Stage 1 mean :",
    r2_series.mean()
)

print(
    "Résidu Stage 1 std :",
    r2_series.std()
)


# ============================================================
# 3. CONSTRUCTION DES SÉQUENCES
# ============================================================

LOOKBACK = 336


values = (
    r2_series
    .to_numpy()
    .astype(np.float32)
)


X_seq = []
y_seq = []
seq_index = []


for i in range(
    LOOKBACK,
    len(values)
):

    X_seq.append(
        values[
            i - LOOKBACK:i
        ]
    )

    y_seq.append(
        values[i]
    )

    seq_index.append(
        r2_series.index[i]
    )


X_seq = np.asarray(
    X_seq,
    dtype=np.float32
)

y_seq = np.asarray(
    y_seq,
    dtype=np.float32
)

seq_index = pd.DatetimeIndex(
    seq_index
)


print(
    "\nSéquences :",
    X_seq.shape
)

print(
    "Période Patch :",
    seq_index[0],
    "->",
    seq_index[-1]
)


# Contrôle de continuité temporelle

expected_index = pd.date_range(
    index_2026[0],
    index_2026[-1],
    freq="30min",
    tz=index_2026.tz,
)

assert index_2026.equals(
    expected_index
)


# ============================================================
# 4. SCALER GELÉ DU BLOC 6
# ============================================================

seq_scaler = joblib.load(
    MODELS_DIR
    / "block6_gru_residual_scaler.joblib"
)


seq_mean = float(
    seq_scaler["mean"]
)

seq_std = float(
    seq_scaler["std"]
)


print(
    "\nFrozen seq mean :",
    seq_mean
)

print(
    "Frozen seq std :",
    seq_std
)


X_seq_scaled = (
    (
        X_seq
        -
        seq_mean
    )
    /
    seq_std
).astype(
    np.float32
)


# batch × 336 × 1

X_seq_scaled = (
    X_seq_scaled[
        ...,
        None
    ]
)


print(
    "X Patch :",
    X_seq_scaled.shape
)


# ============================================================
# 5. ARCHITECTURE EXACTE
# ============================================================

class ResidualPatchTransformer(nn.Module):

    def __init__(
        self,
        seq_len=336,
        patch_len=16,
        stride=8,
        d_model=64,
        n_heads=4,
        n_layers=2,
        d_ff=128,
        dropout=0.10,
    ):

        super().__init__()

        self.patch_len = patch_len
        self.stride = stride

        self.n_patches = (
            (seq_len - patch_len)
            // stride
            + 1
        )

        self.patch_embedding = nn.Linear(
            patch_len,
            d_model
        )

        self.pos_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.n_patches,
                d_model
            )
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=d_ff,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=n_layers,
            )
        )

        self.norm = nn.LayerNorm(
            d_model
        )

        self.head = nn.Sequential(

            nn.Flatten(
                start_dim=1
            ),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                self.n_patches
                * d_model,
                1,
            ),
        )

        nn.init.trunc_normal_(
            self.pos_embedding,
            std=0.02
        )

    def forward(
        self,
        x
    ):

        x = x.squeeze(-1)

        x = x.unfold(
            dimension=1,
            size=self.patch_len,
            step=self.stride,
        )

        x = self.patch_embedding(
            x
        )

        x = (
            x
            +
            self.pos_embedding
        )

        x = self.encoder(
            x
        )

        x = self.norm(
            x
        )

        return self.head(
            x
        )


# ============================================================
# 6. CHARGEMENT DU MODÈLE GELÉ
# ============================================================

patch_model = (
    ResidualPatchTransformer()
    .to(device)
)


PATCH_PATH = (
    MODELS_DIR
    / "block6_residual_patch_transformer.pt"
)


patch_model.load_state_dict(

    torch.load(
        PATCH_PATH,
        map_location=device,
        weights_only=True,
    )

)

patch_model.eval()


n_params = sum(
    p.numel()
    for p
    in patch_model.parameters()
)


print(
    "\nNumber of patches :",
    patch_model.n_patches
)

print(
    "Patch parameters :",
    n_params
)


assert (
    patch_model.n_patches
    == 41
)

assert (
    n_params
    == 73409
)


# ============================================================
# 7. ETA GELÉ
# ============================================================

block6_state = joblib.load(
    MODELS_DIR
    / "block6_final_state.joblib"
)


best_eta_patch = float(
    block6_state[
        "best_eta_patch"
    ]
)


print(
    "Frozen eta Patch :",
    best_eta_patch
)


# Aucun tuning ici

assert np.isclose(
    best_eta_patch,
    0.7
)


# ============================================================
# 8. INFÉRENCE PATCH
# ============================================================

loader = DataLoader(
    torch.from_numpy(
        X_seq_scaled
    ),
    batch_size=256,
    shuffle=False,
)


pred_patch_scaled = []


with torch.inference_mode():

    for xb in loader:

        xb = xb.to(
            device
        )

        pred = patch_model(
            xb
        )

        pred_patch_scaled.append(

            pred
            .float()
            .cpu()
            .numpy()

        )


pred_patch_scaled = (

    np.vstack(
        pred_patch_scaled
    )
    .ravel()

)


# Retour en MW

pred_patch_correction = (

    pred_patch_scaled
    * seq_std
    + seq_mean

)


# ============================================================
# 9. ALIGNEMENT STAGE 1
# ============================================================

stage1_series = pd.Series(
    pred_stage1,
    index=index_2026
)

ridge_series = pd.Series(
    pred_ridge,
    index=index_2026
)

y_series = pd.Series(
    y_true,
    index=index_2026
)


base_stage1_patch = (
    stage1_series
    .loc[seq_index]
    .to_numpy()
)

base_ridge_patch = (
    ridge_series
    .loc[seq_index]
    .to_numpy()
)

y_patch_true = (
    y_series
    .loc[seq_index]
    .to_numpy()
)


# ============================================================
# 10. PRÉDICTION PATCH FINALE
# ============================================================

pred_patch_2026 = (

    base_stage1_patch

    +

    best_eta_patch
    *
    pred_patch_correction

)


# ============================================================
# 11. MÉTRIQUES COMMUNES
# ============================================================

def metrics(
    y,
    pred
):

    mse = mean_squared_error(
        y,
        pred
    )

    return {

        "MSE":
            mse,

        "RMSE":
            np.sqrt(mse),

        "MAE":
            mean_absolute_error(
                y,
                pred
            ),

        "MAPE_%":
            np.mean(
                np.abs(
                    (y - pred)
                    /
                    y
                )
            ) * 100,

        "R2":
            r2_score(
                y,
                pred
            ),
    }


ridge_metrics = metrics(
    y_patch_true,
    base_ridge_patch
)

stage1_metrics = metrics(
    y_patch_true,
    base_stage1_patch
)

patch_metrics = metrics(
    y_patch_true,
    pred_patch_2026
)


comparison = pd.DataFrame(
    [

        {
            "Model":
                "Ridge",

            **ridge_metrics,
        },

        {
            "Model":
                "Ridge + MLP1",

            **stage1_metrics,
        },

        {
            "Model":
                "Ridge + MLP1 + Patch Transformer",

            **patch_metrics,
        },

    ]
)


print(
    "\n"
    + "=" * 72
)

print(
    "FINAL 2026 — COMMON PATCH PERIOD"
)

print(
    "=" * 72
)

print(
    comparison.round(4)
)


# ============================================================
# 12. GAIN DU PATCH
# ============================================================

gain_patch = (

    100

    * (
        stage1_metrics["RMSE"]
        -
        patch_metrics["RMSE"]
    )

    /
    stage1_metrics["RMSE"]

)


print(
    "\nGain Patch vs Stage 1 :",
    round(
        gain_patch,
        3
    ),
    "%"
)


# ============================================================
# 13. SAUVEGARDE
# ============================================================

OUTPUT_PATH = (
    MODELS_DIR
    / "final_2026_patch.joblib"
)


joblib.dump(

    {

        "index":
            seq_index,

        "y_true":
            y_patch_true,

        "pred_ridge":
            base_ridge_patch,

        "pred_stage1":
            base_stage1_patch,

        "patch_correction":
            pred_patch_correction,

        "pred_patch":
            pred_patch_2026,

        "best_eta_patch":
            best_eta_patch,

        "ridge_metrics":
            ridge_metrics,

        "stage1_metrics":
            stage1_metrics,

        "patch_metrics":
            patch_metrics,

        "gain_patch_pct":
            gain_patch,

    },

    OUTPUT_PATH

)


print(
    "\nSauvegardé :",
    OUTPUT_PATH
)
'''


SCRIPT_PATH.write_text(
    textwrap.dedent(
        script_code
    ),
    encoding="utf-8",
)


print(
    "Script créé :",
    SCRIPT_PATH
)


# ============================================================
# EXÉCUTION EXTERNE
# ============================================================

result = subprocess.run(

    [
        "py",
        "-3.11",
        str(SCRIPT_PATH),
    ],

    cwd=str(
        PROJECT_DIR
    ),

    capture_output=True,
    text=True,
)


print(
    result.stdout
)


if result.returncode != 0:

    print(
        "\nERREUR :"
    )

    print(
        result.stderr
    )

    raise RuntimeError(
        "L'inférence Patch Transformer a échoué."
    )


# ============================================================
# RECHARGEMENT
# ============================================================

patch_2026 = joblib.load(

    PROJECT_DIR
    / "models"
    / "final_2026_patch.joblib"

)


comparison = pd.DataFrame(
    [

        {
            "Model":
                "Ridge",

            **patch_2026[
                "ridge_metrics"
            ],
        },

        {
            "Model":
                "Ridge + MLP1",

            **patch_2026[
                "stage1_metrics"
            ],
        },

        {
            "Model":
                "Ridge + MLP1 + Patch Transformer",

            **patch_2026[
                "patch_metrics"
            ],
        },

    ]
)


print(
    "\nTABLEAU FINAL — PÉRIODE COMMUNE"
)

display(
    comparison.round(4)
)


print(
    "\nGain Patch :",
    round(
        patch_2026[
            "gain_patch_pct"
        ],
        3,
    ),
    "%"
)

Script créé : C:\Users\thiba\OneDrive\Bureau\Projet energie\scripts\final_2026_patch_external.py
FINAL 2026 â€” PATCH TRANSFORMER
PyTorch : 2.14.0+cu130
CUDA : True
Device : cuda
GPU : NVIDIA GeForce GTX 1650 Ti with Max-Q Design

Stage 1 : 12575 observations
DÃ©but : 2026-01-08 00:00:00+00:00
Fin : 2026-09-26 23:00:00+00:00

RÃ©sidu Stage 1 mean : 12.129212975318804
RÃ©sidu Stage 1 std : 261.69660351975557

SÃ©quences : (12239, 336)
PÃ©riode Patch : 2026-01-15 00:00:00+00:00 -> 2026-09-26 23:00:00+00:00

Frozen seq mean : 4.202218055725098
Frozen seq std : 170.05972290039062
X Patch : (12239, 336, 1)

Number of patches : 41
Patch parameters : 73409
Frozen eta Patch : 0.7000000000000001

FINAL 2026 â€” COMMON PATCH PERIOD
                              Model         MSE  ...  MAPE_%      R2
0                             Ridge  81529.4536  ...  0.4480  0.9990
1                      Ridge + MLP1  69332.5166  ...  0.4134  0.9991
2  Ridge + MLP1 + Patch Transformer  65601.0588  ...  0.3994 

,Model,MSE,RMSE,MAE,MAPE_%,R2
0,Ridge,81529.4536,285.5336,208.8841,0.4480,0.9990
1,Ridge + MLP1,69332.5166,263.3107,193.0551,0.4134,0.9991
2,Ridge + MLP1 + Patch Transformer,65601.0588,256.1270,186.3006,0.3994,0.9992



Gain Patch : 2.728 %


In [16]:
# ============================================================
# FINAL 2026 — REBUILD COMPATIBLE EVENT EXPERT
# + VALIDATION CONTRE LE MODÈLE GELÉ 2025
# + ÉVALUATION FINALE 2026
# ============================================================

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from dateutil.easter import easter

from sklearn.ensemble import HistGradientBoostingRegressor

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)


# ============================================================
# 1. CHEMINS
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\thiba\OneDrive\Bureau\Projet energie"
)

MODELS_DIR = (
    PROJECT_DIR
    / "models"
)


# ============================================================
# 2. JOURS FÉRIÉS FRANÇAIS
# EXACTEMENT COMME BLOC 7
# ============================================================

def french_holidays(year):

    easter_day = pd.Timestamp(
        easter(year)
    )

    return {

        pd.Timestamp(f"{year}-01-01"):
            "New Year",

        easter_day
        + pd.Timedelta(days=1):
            "Easter Monday",

        pd.Timestamp(f"{year}-05-01"):
            "Labour Day",

        pd.Timestamp(f"{year}-05-08"):
            "Victory 1945",

        easter_day
        + pd.Timedelta(days=39):
            "Ascension",

        easter_day
        + pd.Timedelta(days=50):
            "Whit Monday",

        pd.Timestamp(f"{year}-07-14"):
            "Bastille Day",

        pd.Timestamp(f"{year}-08-15"):
            "Assumption",

        pd.Timestamp(f"{year}-11-01"):
            "All Saints",

        pd.Timestamp(f"{year}-11-11"):
            "Armistice",

        pd.Timestamp(f"{year}-12-25"):
            "Christmas",
    }


holiday_dict = {}


for year in range(
    2013,
    2027
):

    holiday_dict.update(
        french_holidays(
            year
        )
    )


holiday_dates = set(
    holiday_dict.keys()
)


print(
    "Jours fériés disponibles :",
    len(holiday_dates)
)


# ============================================================
# 3. DST
# ============================================================

dst_2024 = [

    pd.Timestamp(
        "2024-03-31 01:00:00",
        tz="UTC",
    ),

    pd.Timestamp(
        "2024-10-27 01:00:00",
        tz="UTC",
    ),
]


dst_2025 = [

    pd.Timestamp(
        "2025-03-30 01:00:00",
        tz="UTC",
    ),

    pd.Timestamp(
        "2025-10-26 01:00:00",
        tz="UTC",
    ),
]


dst_2026 = [

    pd.Timestamp(
        "2026-03-29 01:00:00",
        tz="UTC",
    ),

    pd.Timestamp(
        "2026-10-25 01:00:00",
        tz="UTC",
    ),
]


# ============================================================
# 4. CALENDAR EXOG
# ============================================================

def build_calendar_exog(
    index,
    dst_transitions,
):

    index = pd.DatetimeIndex(
        index
    )

    local = index.tz_convert(
        "Europe/Paris"
    )

    dates = pd.DatetimeIndex(
        local.date
    )

    df = pd.DataFrame(
        index=index
    )


    # Heure locale

    hour = (
        local.hour
        +
        local.minute / 60
    )


    df["hour_sin"] = np.sin(
        2 * np.pi * hour / 24
    )

    df["hour_cos"] = np.cos(
        2 * np.pi * hour / 24
    )


    # Jour de semaine

    dow = local.dayofweek


    df["dow_sin"] = np.sin(
        2 * np.pi * dow / 7
    )

    df["dow_cos"] = np.cos(
        2 * np.pi * dow / 7
    )


    # Jour férié

    df["is_holiday"] = [

        int(
            pd.Timestamp(d)
            in holiday_dates
        )

        for d in dates
    ]


    # Veille de jour férié

    df["holiday_eve"] = [

        int(
            pd.Timestamp(d)
            + pd.Timedelta(days=1)
            in holiday_dates
        )

        for d in dates
    ]


    # Lendemain de jour férié

    df["after_holiday"] = [

        int(
            pd.Timestamp(d)
            - pd.Timedelta(days=1)
            in holiday_dates
        )

        for d in dates
    ]


    # Noël / Nouvel An

    df["year_end_period"] = [

        int(
            (
                d.month == 12
                and d.day >= 23
            )
            or
            (
                d.month == 1
                and d.day <= 2
            )
        )

        for d in local
    ]


    # DST +/-24 h

    dst_flag = np.zeros(
        len(index),
        dtype=int
    )


    for transition in dst_transitions:

        distance = np.abs(

            (
                index
                -
                transition
            )

            /
            pd.Timedelta(
                hours=1
            )
        )


        dst_flag[
            distance <= 24
        ] = 1


    df["dst_window_24h"] = (
        dst_flag
    )


    return df.astype(float)


# ============================================================
# 5. CHARGEMENT STAGE 1 2024-2025
# ============================================================

stage1_state = joblib.load(

    MODELS_DIR
    / "block6_stage1_full_2024_2025.joblib"

)


idx_2024 = pd.DatetimeIndex(
    stage1_state[
        "index_2024"
    ]
)

r2_2024 = np.asarray(
    stage1_state[
        "r2_2024"
    ],
    dtype=float,
)


idx_2025_full = pd.DatetimeIndex(
    stage1_state[
        "index_2025"
    ]
)

pred_stage1_2025_full = np.asarray(
    stage1_state[
        "pred_stage1_2025"
    ],
    dtype=float,
)


print(
    "\n2024 :",
    len(idx_2024)
)

print(
    "2025 Stage 1 :",
    len(idx_2025_full)
)


# ============================================================
# 6. RECONSTRUCTION DES FEATURES
# ============================================================

event_X_2024 = build_calendar_exog(

    idx_2024,

    dst_2024
    +
    dst_2025,

)


event_X_2025 = build_calendar_exog(

    idx_2025_full,

    dst_2024
    +
    dst_2025,

)


print(
    "\nFeatures :",
    list(
        event_X_2024.columns
    )
)


# ============================================================
# 7. TRAINING MASK ORIGINAL
# ============================================================

cut = pd.Timestamp(

    "2024-10-01",

    tz=idx_2024.tz,
)


expert_train_mask = (
    idx_2024
    <
    cut
)


print(
    "\nObservations entraînement Expert :",
    expert_train_mask.sum()
)


# ============================================================
# 8. EVENT MASK ORIGINAL
# ============================================================

event_columns = [

    "is_holiday",

    "holiday_eve",

    "after_holiday",

    "year_end_period",

    "dst_window_24h",
]


event_mask_2024 = (

    event_X_2024[
        event_columns
    ]

    .max(axis=1)

    .astype(bool)
)


# ============================================================
# 9. POIDS ORIGINAUX
# ============================================================

sample_weight = np.ones(
    len(event_X_2024)
)


sample_weight[
    event_mask_2024.to_numpy()
] = 8.0


print(
    "Observations événement 2024 :",
    int(
        event_mask_2024.sum()
    )
)


# ============================================================
# 10. RECONSTRUCTION EXACTE DE L'EXPERT
# ============================================================

event_expert = (

    HistGradientBoostingRegressor(

        learning_rate=0.05,

        max_iter=150,

        max_depth=3,

        l2_regularization=10.0,

        early_stopping=True,

        random_state=42,
    )
)


event_expert.fit(

    event_X_2024.loc[
        expert_train_mask
    ],

    r2_2024[
        expert_train_mask
    ],

    sample_weight=sample_weight[
        expert_train_mask
    ],
)


print(
    "\nEvent Expert reconstruit."
)

print(
    "Iterations :",
    event_expert.n_iter_
)


# ============================================================
# 11. SAUVEGARDE COMPATIBLE
# ============================================================

REBUILT_MODEL_PATH = (

    MODELS_DIR
    / "block7_event_expert_rebuilt_compat.joblib"

)


joblib.dump(

    event_expert,

    REBUILT_MODEL_PATH

)


print(
    "Modèle compatible sauvegardé :",
    REBUILT_MODEL_PATH
)


# ============================================================
# 12. VALIDATION CONTRE LA PRÉDICTION GELÉE 2025
# ============================================================

block6_state = joblib.load(

    MODELS_DIR
    / "block6_final_state.joblib"

)


refined_state = joblib.load(

    MODELS_DIR
    / "block7_refined_event_expert.joblib"

)


idx_patch_2025 = pd.DatetimeIndex(

    block6_state[
        "eval_index_2025"
    ]
)


pred_patch_2025 = np.asarray(

    block6_state[
        "pred_patch_2025"
    ],

    dtype=float,
)


stage1_2025_series = pd.Series(

    pred_stage1_2025_full,

    index=idx_2025_full,
)


patch_2025_series = pd.Series(

    pred_patch_2025,

    index=idx_patch_2025,
)


common_2025 = (

    patch_2025_series.index

    .intersection(
        stage1_2025_series.index
    )
)


stage1_common = (

    stage1_2025_series

    .loc[
        common_2025
    ]

    .to_numpy()
)


patch_common = (

    patch_2025_series

    .loc[
        common_2025
    ]

    .to_numpy()
)


patch_correction_2025 = (

    patch_common
    -
    stage1_common
)


expert_correction_2025 = (

    event_expert.predict(

        event_X_2025.loc[
            common_2025
        ]

    )
)


refined_event_columns = [

    "is_holiday",

    "year_end_period",

    "after_holiday",
]


lambda_event = 0.5


refined_mask_2025 = (

    event_X_2025

    .loc[
        common_2025,
        refined_event_columns,
    ]

    .max(axis=1)

    .astype(bool)

    .to_numpy()
)


pred_refined_rebuilt_2025 = (

    patch_common.copy()
)


refined_correction_2025 = (

    (1 - lambda_event)

    *
    patch_correction_2025[
        refined_mask_2025
    ]

    +

    lambda_event

    *
    expert_correction_2025[
        refined_mask_2025
    ]
)


pred_refined_rebuilt_2025[
    refined_mask_2025
] = (

    stage1_common[
        refined_mask_2025
    ]

    +

    refined_correction_2025
)


# ============================================================
# 13. COMPARAISON AVEC PRÉDICTION 2025 SAUVEGARDÉE
# ============================================================

saved_index_2025 = pd.DatetimeIndex(

    refined_state[
        "index_2025"
    ]
)


saved_pred_2025 = np.asarray(

    refined_state[
        "prediction_2025"
    ],

    dtype=float,
)


assert common_2025.equals(
    saved_index_2025
)


prediction_diff = (

    pred_refined_rebuilt_2025

    -

    saved_pred_2025
)


validation_mae = np.mean(
    np.abs(
        prediction_diff
    )
)


validation_rmse = np.sqrt(
    np.mean(
        prediction_diff ** 2
    )
)


validation_max = np.max(
    np.abs(
        prediction_diff
    )
)


print(
    "\n"
    + "=" * 75
)

print(
    "VALIDATION RECONSTRUCTION 2025"
)

print(
    "=" * 75
)


print(
    "MAE différence :",
    validation_mae
)

print(
    "RMSE différence :",
    validation_rmse
)

print(
    "Max différence :",
    validation_max
)


# Critère strict :
# la reconstruction doit être pratiquement identique.

REPRODUCTION_OK = (
    validation_rmse
    <
    1e-6
)


print(
    "\nReproduction exacte :",
    REPRODUCTION_OK
)


if not REPRODUCTION_OK:

    raise RuntimeError(

        "La reconstruction de l'Event Expert ne "
        "reproduit pas exactement le modèle gelé 2025. "
        "On n'applique donc PAS ce modèle à 2026."

    )


# ============================================================
# 14. CHARGEMENT PATCH 2026
# ============================================================

patch_2026 = joblib.load(

    MODELS_DIR
    / "final_2026_patch.joblib"

)


index_2026 = pd.DatetimeIndex(

    patch_2026[
        "index"
    ]
)


y_true = np.asarray(

    patch_2026[
        "y_true"
    ],

    dtype=float,
)


pred_ridge = np.asarray(

    patch_2026[
        "pred_ridge"
    ],

    dtype=float,
)


pred_stage1 = np.asarray(

    patch_2026[
        "pred_stage1"
    ],

    dtype=float,
)


pred_patch = np.asarray(

    patch_2026[
        "pred_patch"
    ],

    dtype=float,
)


# ============================================================
# 15. FEATURES EVENT 2026
# ============================================================

event_X_2026 = build_calendar_exog(

    index_2026,

    dst_2026,
)


assert (

    list(
        event_X_2026.columns
    )

    ==

    list(
        event_expert.feature_names_in_
    )
)


# ============================================================
# 16. CORRECTIONS 2026
# ============================================================

patch_correction_2026 = (

    pred_patch
    -
    pred_stage1
)


expert_correction_2026 = (

    event_expert.predict(
        event_X_2026
    )
)


refined_mask_2026 = (

    event_X_2026[
        refined_event_columns
    ]

    .max(axis=1)

    .astype(bool)

    .to_numpy()
)


print(
    "\nObservations refined gate 2026 :",
    int(
        refined_mask_2026.sum()
    )
)


print(
    "Part refined gate :",
    100
    *
    refined_mask_2026.mean(),
    "%"
)


# ============================================================
# 17. MODÈLE FINAL 2026
# ============================================================

pred_final_2026 = (
    pred_patch.copy()
)


refined_correction_2026 = (

    (1 - lambda_event)

    *
    patch_correction_2026[
        refined_mask_2026
    ]

    +

    lambda_event

    *
    expert_correction_2026[
        refined_mask_2026
    ]
)


pred_final_2026[
    refined_mask_2026
] = (

    pred_stage1[
        refined_mask_2026
    ]

    +

    refined_correction_2026
)


# ============================================================
# 18. MÉTRIQUES
# ============================================================

def regression_metrics(
    y,
    pred,
):

    mse = mean_squared_error(
        y,
        pred
    )

    rmse = np.sqrt(
        mse
    )

    mae = mean_absolute_error(
        y,
        pred
    )


    # Erreur absolue moyenne en %

    mape = (

        np.mean(

            np.abs(

                (y - pred)

                /
                y
            )
        )

        * 100
    )


    # RMSE ramené au niveau moyen de consommation

    nrmse = (

        100

        *
        rmse

        /
        np.mean(y)
    )


    return {

        "MSE":
            mse,

        "RMSE":
            rmse,

        "MAE":
            mae,

        "MAPE_%":
            mape,

        "NRMSE_%":
            nrmse,

        "R2":
            r2_score(
                y,
                pred
            ),
    }


ridge_metrics = regression_metrics(
    y_true,
    pred_ridge,
)


stage1_metrics = regression_metrics(
    y_true,
    pred_stage1,
)


patch_metrics = regression_metrics(
    y_true,
    pred_patch,
)


final_metrics = regression_metrics(
    y_true,
    pred_final_2026,
)


# ============================================================
# 19. TABLEAU FINAL
# ============================================================

comparison = pd.DataFrame(

    [

        {
            "Model":
                "Ridge",

            **ridge_metrics,
        },

        {
            "Model":
                "Ridge + MLP1",

            **stage1_metrics,
        },

        {
            "Model":
                "Ridge + MLP1 + Patch",

            **patch_metrics,
        },

        {
            "Model":
                (
                    "FINAL — Ridge + MLP1 + "
                    "Patch + Refined Event Expert"
                ),

            **final_metrics,
        },

    ]
)


print(
    "\n"
    + "=" * 85
)

print(
    "ÉVALUATION FINALE — TEST INDÉPENDANT 2026"
)

print(
    "=" * 85
)


display(
    comparison.round(4)
)


# ============================================================
# 20. GAINS
# ============================================================

gain_patch_vs_ridge = (

    100

    *
    (
        ridge_metrics[
            "RMSE"
        ]

        -

        patch_metrics[
            "RMSE"
        ]
    )

    /
    ridge_metrics[
        "RMSE"
    ]
)


gain_final_vs_patch = (

    100

    *
    (
        patch_metrics[
            "RMSE"
        ]

        -

        final_metrics[
            "RMSE"
        ]
    )

    /
    patch_metrics[
        "RMSE"
    ]
)


gain_final_vs_ridge = (

    100

    *
    (
        ridge_metrics[
            "RMSE"
        ]

        -

        final_metrics[
            "RMSE"
        ]
    )

    /
    ridge_metrics[
        "RMSE"
    ]
)


print(
    "\nGain Patch vs Ridge :",
    round(
        gain_patch_vs_ridge,
        3
    ),
    "%"
)


print(
    "Gain Event Expert vs Patch :",
    round(
        gain_final_vs_patch,
        3
    ),
    "%"
)


print(
    "Gain modèle final vs Ridge :",
    round(
        gain_final_vs_ridge,
        3
    ),
    "%"
)


# ============================================================
# 21. INTERPRÉTATION ERREUR %
# ============================================================

print(
    "\n"
    + "=" * 85
)

print(
    "ERREUR FINALE EN POURCENTAGE"
)

print(
    "=" * 85
)


print(
    "MAPE :",
    round(
        final_metrics[
            "MAPE_%"
        ],
        4
    ),
    "%"
)


print(
    "NRMSE :",
    round(
        final_metrics[
            "NRMSE_%"
        ],
        4
    ),
    "%"
)


print(
    "\nInterprétation :"
)


print(
    "MAPE = écart absolu moyen entre "
    "la prédiction et la consommation réelle."
)


print(
    "NRMSE = erreur quadratique globale "
    "rapportée au niveau moyen de consommation."
)


# ============================================================
# 22. SAUVEGARDE FINALE
# ============================================================

FINAL_PATH = (

    MODELS_DIR
    / "final_2026_complete_evaluation.joblib"

)


joblib.dump(

    {

        "model_name":
            (
                "Ridge + MLP1 + "
                "Patch Transformer + "
                "Refined Event Expert"
            ),

        "index":
            index_2026,

        "y_true":
            y_true,

        "pred_ridge":
            pred_ridge,

        "pred_stage1":
            pred_stage1,

        "pred_patch":
            pred_patch,

        "prediction_final":
            pred_final_2026,

        "event_mask":
            refined_mask_2026,

        "event_columns":
            refined_event_columns,

        "event_lambda":
            lambda_event,

        "ridge_metrics":
            ridge_metrics,

        "stage1_metrics":
            stage1_metrics,

        "patch_metrics":
            patch_metrics,

        "final_metrics":
            final_metrics,

        "gain_final_vs_patch_pct":
            gain_final_vs_patch,

        "gain_final_vs_ridge_pct":
            gain_final_vs_ridge,

        "rebuild_validation_2025": {

            "mae_difference":
                validation_mae,

            "rmse_difference":
                validation_rmse,

            "max_difference":
                validation_max,
        },

    },

    FINAL_PATH,
)


print(
    "\nSauvegardé :",
    FINAL_PATH
)

Jours fériés disponibles : 154

2024 : 17568
2025 Stage 1 : 17517

Features : ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_holiday', 'holiday_eve', 'after_holiday', 'year_end_period', 'dst_window_24h']

Observations entraînement Expert : 13152
Observations événement 2024 : 1867

Event Expert reconstruit.
Iterations : 35
Modèle compatible sauvegardé : C:\Users\thiba\OneDrive\Bureau\Projet energie\models\block7_event_expert_rebuilt_compat.joblib

VALIDATION RECONSTRUCTION 2025
MAE différence : 0.0
RMSE différence : 0.0
Max différence : 0.0

Reproduction exacte : True

Observations refined gate 2026 : 672
Part refined gate : 5.49064466051148 %

ÉVALUATION FINALE — TEST INDÉPENDANT 2026


,Model,MSE,RMSE,MAE,MAPE_%,NRMSE_%,R2
0,Ridge,81529.4536,285.5336,208.8841,0.4480,0.5899,0.9990
1,Ridge + MLP1,69332.5166,263.3107,193.0551,0.4134,0.5440,0.9991
2,Ridge + MLP1 + Patch,65601.0588,256.1270,186.3006,0.3994,0.5292,0.9992
3,FINAL — Ridge + MLP1 + Patch + Refined Event E...,65489.6422,255.9094,186.1091,0.3988,0.5287,0.9992



Gain Patch vs Ridge : 10.299 %
Gain Event Expert vs Patch : 0.085 %
Gain modèle final vs Ridge : 10.375 %

ERREUR FINALE EN POURCENTAGE
MAPE : 0.3988 %
NRMSE : 0.5287 %

Interprétation :
MAPE = écart absolu moyen entre la prédiction et la consommation réelle.
NRMSE = erreur quadratique globale rapportée au niveau moyen de consommation.

Sauvegardé : C:\Users\thiba\OneDrive\Bureau\Projet energie\models\final_2026_complete_evaluation.joblib


## Analyse finale — Généralisation temporelle sur le test indépendant 2026

L'objectif de cette dernière analyse est d'évaluer la capacité du pipeline final à se généraliser à une période temporelle plus récente, jamais utilisée pour l'entraînement, la sélection des modèles ou le réglage des hyperparamètres.

Le modèle final est composé de :

- **Ridge Regression** pour la structure linéaire haute dimensionnelle ;
- **Residual MLP** pour corriger les erreurs non linéaires du Ridge ;
- **Patch Transformer** pour exploiter la dépendance temporelle des résidus ;
- **Refined Event Expert** pour traiter certains régimes calendaires spécifiques.

Tous les composants, hyperparamètres, coefficients de correction et règles de gating ont été fixés avant l'évaluation sur les données 2026.

Le jeu de test final couvre la période du **15 janvier au 26 septembre 2026**, soit **12 239 observations à pas de 30 minutes**.

In [17]:
# ============================================================
# FINAL ANALYSIS — 2025 vs INDEPENDENT 2026
# Temporal Generalization Analysis
# ============================================================

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)


PROJECT_DIR = Path(
    r"C:\Users\thiba\OneDrive\Bureau\Projet energie"
)

MODELS_DIR = PROJECT_DIR / "models"


# ============================================================
# 1. LOAD 2025 / 2026 FINAL STATES
# ============================================================

state_2025 = joblib.load(
    MODELS_DIR
    / "block7_final_state.joblib"
)

state_2026 = joblib.load(
    MODELS_DIR
    / "final_2026_complete_evaluation.joblib"
)


y_2025 = np.asarray(
    state_2025["y_2025"],
    dtype=float,
)

pred_2025 = np.asarray(
    state_2025["prediction_2025"],
    dtype=float,
)

idx_2025 = pd.DatetimeIndex(
    state_2025["index_2025"]
)


y_2026 = np.asarray(
    state_2026["y_true"],
    dtype=float,
)

pred_2026 = np.asarray(
    state_2026["prediction_final"],
    dtype=float,
)

idx_2026 = pd.DatetimeIndex(
    state_2026["index"]
)


# ============================================================
# 2. METRICS
# ============================================================

def full_metrics(
    y,
    pred,
):

    residual = y - pred

    mse = mean_squared_error(
        y,
        pred
    )

    rmse = np.sqrt(
        mse
    )

    mae = mean_absolute_error(
        y,
        pred
    )

    mape = (
        np.mean(
            np.abs(
                residual / y
            )
        )
        * 100
    )

    nrmse = (
        100
        * rmse
        / np.mean(y)
    )

    return {

        "N":
            len(y),

        "Mean_target":
            np.mean(y),

        "RMSE":
            rmse,

        "MAE":
            mae,

        "MAPE_%":
            mape,

        "NRMSE_%":
            nrmse,

        "R2":
            r2_score(
                y,
                pred
            ),

        "Bias":
            np.mean(
                residual
            ),

        "Residual_std":
            np.std(
                residual
            ),

        "Median_abs_error":
            np.median(
                np.abs(
                    residual
                )
            ),

        "P90_abs_error":
            np.quantile(
                np.abs(
                    residual
                ),
                0.90
            ),

        "P95_abs_error":
            np.quantile(
                np.abs(
                    residual
                ),
                0.95
            ),

        "P99_abs_error":
            np.quantile(
                np.abs(
                    residual
                ),
                0.99
            ),
    }


metrics_2025 = full_metrics(
    y_2025,
    pred_2025,
)

metrics_2026 = full_metrics(
    y_2026,
    pred_2026,
)


comparison = pd.DataFrame(

    [
        {
            "Period":
                "2025",

            **metrics_2025,
        },

        {
            "Period":
                "2026 independent test",

            **metrics_2026,
        },

    ]

).set_index(
    "Period"
)


print(
    "=" * 90
)

print(
    "FINAL MODEL — TEMPORAL GENERALIZATION"
)

print(
    "=" * 90
)

display(
    comparison.round(4)
)


# ============================================================
# 3. RELATIVE CHANGES
# ============================================================

metrics_to_compare = [

    "RMSE",

    "MAE",

    "MAPE_%",

    "NRMSE_%",

    "Median_abs_error",

    "P90_abs_error",

    "P95_abs_error",

    "P99_abs_error",
]


relative_changes = {}


for metric in metrics_to_compare:

    value_2025 = (
        metrics_2025[
            metric
        ]
    )

    value_2026 = (
        metrics_2026[
            metric
        ]
    )

    change = (

        100

        * (
            value_2026
            -
            value_2025
        )

        /
        value_2025
    )

    relative_changes[
        metric
    ] = change


relative_df = pd.DataFrame(

    {
        "2025":
            [
                metrics_2025[m]
                for m
                in metrics_to_compare
            ],

        "2026":
            [
                metrics_2026[m]
                for m
                in metrics_to_compare
            ],

        "Change_%":
            [
                relative_changes[m]
                for m
                in metrics_to_compare
            ],
    },

    index=metrics_to_compare,
)


print(
    "\n"
    + "=" * 90
)

print(
    "RELATIVE CHANGE — 2025 → 2026"
)

print(
    "=" * 90
)

display(
    relative_df.round(4)
)


# ============================================================
# 4. RESIDUAL DISTRIBUTION
# ============================================================

residual_2025 = (
    y_2025
    -
    pred_2025
)

residual_2026 = (
    y_2026
    -
    pred_2026
)


residual_summary = pd.DataFrame(

    {

        "2025": [

            residual_2025.mean(),

            np.median(
                residual_2025
            ),

            residual_2025.std(),

            residual_2025.min(),

            residual_2025.max(),

            np.mean(
                np.abs(
                    residual_2025
                )
            ),

        ],

        "2026": [

            residual_2026.mean(),

            np.median(
                residual_2026
            ),

            residual_2026.std(),

            residual_2026.min(),

            residual_2026.max(),

            np.mean(
                np.abs(
                    residual_2026
                )
            ),

        ],

    },

    index=[

        "Mean residual",

        "Median residual",

        "Std residual",

        "Min residual",

        "Max residual",

        "Mean absolute residual",

    ],

)


print(
    "\n"
    + "=" * 90
)

print(
    "RESIDUAL DISTRIBUTION"
)

print(
    "=" * 90
)

display(
    residual_summary.round(4)
)


# ============================================================
# 5. MODEL PROGRESSION 2026
# ============================================================

model_progression = pd.DataFrame(

    [

        {
            "Model":
                "Ridge",

            **state_2026[
                "ridge_metrics"
            ],
        },

        {
            "Model":
                "Ridge + MLP1",

            **state_2026[
                "stage1_metrics"
            ],
        },

        {
            "Model":
                "Ridge + MLP1 + Patch",

            **state_2026[
                "patch_metrics"
            ],
        },

        {
            "Model":
                "Final + Event Expert",

            **state_2026[
                "final_metrics"
            ],
        },

    ]

).set_index(
    "Model"
)


print(
    "\n"
    + "=" * 90
)

print(
    "MODEL PROGRESSION — INDEPENDENT 2026"
)

print(
    "=" * 90
)

display(
    model_progression.round(4)
)


# ============================================================
# 6. SAVE
# ============================================================

analysis_state = {

    "metrics_2025":
        metrics_2025,

    "metrics_2026":
        metrics_2026,

    "relative_changes":
        relative_changes,

    "comparison":
        comparison,

    "residual_summary":
        residual_summary,

    "model_progression_2026":
        model_progression,

    "period_2025": (
        idx_2025.min(),
        idx_2025.max(),
    ),

    "period_2026": (
        idx_2026.min(),
        idx_2026.max(),
    ),
}


joblib.dump(

    analysis_state,

    MODELS_DIR
    / "final_temporal_generalization_analysis.joblib"

)


comparison.to_csv(

    PROJECT_DIR
    / "data_processed"
    / "final_2025_2026_comparison.csv"

)


print(
    "\nAnalyse sauvegardée."
)

FINAL MODEL — TEMPORAL GENERALIZATION


,N,Mean_target,RMSE,MAE,MAPE_%,NRMSE_%,R2,Bias,Residual_std,Median_abs_error,P90_abs_error,P95_abs_error,P99_abs_error
Period,,,,,,,,,,,,,
2025,17517,50668.4393,176.9426,127.8516,0.2576,0.3492,0.9997,-2.2487,176.9283,99.7523,264.42,328.2916,520.1375
2026 independent test,12239,48401.5355,255.9094,186.1091,0.3988,0.5287,0.9992,9.0408,255.7497,141.1060,393.30,508.2552,827.6793



RELATIVE CHANGE — 2025 → 2026


,2025,2026,Change_%
RMSE,176.9426,255.9094,44.6285
MAE,127.8516,186.1091,45.5665
MAPE_%,0.2576,0.3988,54.8219
NRMSE_%,0.3492,0.5287,51.4023
Median_abs_error,99.7523,141.1060,41.4564
P90_abs_error,264.4200,393.3000,48.7406
P95_abs_error,328.2916,508.2552,54.8182
P99_abs_error,520.1375,827.6793,59.1270



RESIDUAL DISTRIBUTION


,2025,2026
Mean residual,-2.2487,9.0408
Median residual,-1.7326,8.0195
Std residual,176.9283,255.7497
Min residual,-2847.2220,-2471.5487
Max residual,2386.1167,2320.2978
Mean absolute residual,127.8516,186.1091



MODEL PROGRESSION — INDEPENDENT 2026


,MSE,RMSE,MAE,MAPE_%,NRMSE_%,R2
Model,,,,,,
Ridge,81529.4536,285.5336,208.8841,0.4480,0.5899,0.9990
Ridge + MLP1,69332.5166,263.3107,193.0551,0.4134,0.5440,0.9991
Ridge + MLP1 + Patch,65601.0588,256.1270,186.3006,0.3994,0.5292,0.9992
Final + Event Expert,65489.6422,255.9094,186.1091,0.3988,0.5287,0.9992



Analyse sauvegardée.


## Interprétation des performances 2025–2026

Le modèle final obtenait en 2025 :

$$
RMSE_{2025} = 176.94 \text{ MW}
$$

$$
MAE_{2025} = 127.85 \text{ MW}
$$

$$
MAPE_{2025} = 0.2576\%
$$

Sur le test indépendant 2026, les performances deviennent :

$$
RMSE_{2026} = 255.91 \text{ MW}
$$

$$
MAE_{2026} = 186.11 \text{ MW}
$$

$$
MAPE_{2026} = 0.3988\%
$$

$$
NRMSE_{2026} = 0.5287\%
$$

Entre 2025 et 2026 :

- le RMSE augmente de **44.63 %** ;
- le MAE augmente de **45.57 %

## Apport des différents composants sur le test indépendant 2026

Les performances montrent que les différentes améliorations du pipeline restent utiles sur les données 2026.

Le modèle Ridge de référence obtient :

$$
RMSE_{Ridge} = 285.53 \text{ MW}
$$

L'ajout du Residual MLP réduit le RMSE à :

$$
RMSE_{Ridge+MLP} = 263.31 \text{ MW}
$$

Le Patch Transformer améliore encore le résultat :

$$
RMSE_{Patch} = 256.13 \text{ MW}
$$

Enfin, le modèle complet atteint :

$$
\boxed{RMSE_{Final} = 255.91 \text{ MW}}
$$

Le gain du modèle final par rapport au Ridge est donc :

$$
\frac{285.53 - 255.91}{285.53} \times 100
\approx 10.38\%
$$

soit une réduction du RMSE d'environ :

$$
29.62 \text{ MW}
$$

Le modèle final obtient également :

$$
\boxed{MAE = 186.11 \text{ MW}}
$$

$$
\boxed{MAPE = 0.3988\%}
$$

$$
\boxed{NRMSE = 0.5287\%}
$$

avec :

$$
R^2 \approx 0.9992
$$

Le MAPE de **0.3988 %** signifie que l'écart absolu entre la prédiction et la consommation réelle représente en moyenne environ **0.40 % de la consommation observée**.

Le Refined Event Expert apporte une amélioration globale faible :

$$
256.13
\rightarrow
255.91 \text{ MW}
$$

soit environ :

$$
0.085\%
$$

de gain supplémentaire sur le RMSE.

Cela est cohérent avec son rôle spécialisé, puisqu'il intervient seulement sur environ **5.49 % des observations 2026**.

Le résultat principal est que les améliorations développées avant l'ouverture du test 2026 restent efficaces sur des données futures : le pipeline complet conserve un avantage significatif par rapport au Ridge de référence.

# Conclusion finale

Cette étude a conduit au développement d'un pipeline hybride de prévision de consommation électrique combinant méthodes statistiques, machine learning et deep learning.

Le modèle final est composé de :

$$
\boxed{
Ridge
+
Residual\ MLP
+
Patch\ Transformer
+
Refined\ Event\ Expert
}
$$

L'évaluation finale a été réalisée sur **12 239 observations indépendantes de 2026**, couvrant la période du **15 janvier au 26 septembre 2026**.

Ces données n'ont pas été utilisées pour l'entraînement, la sélection des modèles ou le réglage des hyperparamètres.

Les performances finales sont :

$$
\boxed{RMSE = 255.91 \text{ MW}}
$$

$$
\boxed{MAE = 186.11 \text{ MW}}
$$

$$
\boxed{MAPE = 0.3988\%}
$$

$$
\boxed{NRMSE = 0.5287\%}
$$

avec :

$$
R^2 \approx 0.9992
$$

Par rapport au Ridge de référence, le pipeline complet réduit le RMSE de :

$$
\boxed{10.38\%}
$$

Les performances sont moins bonnes qu'en 2025, avec une augmentation importante de la dispersion des résidus et des erreurs extrêmes.

Cependant, le Residual MLP et le Patch Transformer conservent un gain important sur les données futures, ce qui montre que les améliorations obtenues pendant le développement ne disparaissent pas lors du passage au test indépendant.

Le Refined Event Expert apporte un gain global plus faible, mais son rôle est spécialisé sur un nombre limité de situations calendaires.

La prochaine étape consiste à identifier plus précisément les causes de la dégradation observée en 2026, notamment selon :

- le mois ;
- l'heure de la journée ;
- le jour de la semaine ;
- la température ;
- les événements calendaires ;
- les 1 % d'erreurs les plus importantes.